# S4. Семантичний шар: ембединги bge-m3, семантичні ребра, копії золотих, досяжність

**Що робить ноутбук.** Бере з вашого Google Drive файли S1 (`MyDrive/narratives-thesis/data/raw/`), кодує моделлю `BAAI/bge-m3`
приблизно 108 тисяч повідомлень (412 золотих, уся розмічена партія 81 584 і кінці семантичних ребер набору), порівнює наш косинус з оцінками
ребер набору, будує власні семантичні ребра (hnswlib), шукає точні копії золотих повідомлень, рахує досяжність між групами каналів
для форвардів і семантики, малює UMAP і звіряє `forward_edges` з таблицею `channel_flow_edges` (залишок S3).
Результати зберігаються в `MyDrive/narratives-thesis/results/04_semantic/`, ембединги (float16, близько 220 МБ) у
`MyDrive/narratives-thesis/data/interim/embeddings/`. Код репозиторію вбудований у ноутбук, окремо нічого ставити не потрібно.

**Що зробити вам.**
1. Runtime → Change runtime type → **T4 GPU** (без GPU ноутбук зупиниться на початку з поясненням).
2. Runtime → Run all, дозволити доступ до Drive.
3. Коли все завершиться, надішліть мені файл `MyDrive/narratives-thesis/results/04_semantic.zip` (як минулого разу архів).

Орієнтовний час 30-45 хвилин (модель завантажується близько 2 ГБ, кодування 10-15 хвилин, UMAP кілька хвилин). Потрібно приблизно 8-10 ГБ оперативної
пам'яті. Якщо сеанс обірветься, просто запустіть знову: ембединги кешуються на Drive.


In [ ]:
# 1. Drive, залежності, GPU, шляхи
import hashlib, json, os, subprocess, sys, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/narratives-thesis")
    REPO = Path("/content/repo")
except ImportError:  # запуск поза Colab (тестовий)
    BASE = Path("./narratives-thesis").resolve()
    REPO = Path("./repo").resolve()

RAW = BASE / "data" / "raw"
OUT = BASE / "results" / "04_semantic"
INTERIM = BASE / "data" / "interim" / "embeddings"
assert RAW.exists(), f"Немає {RAW}. Спершу запустіть ноутбук S1."
OUT.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "hnswlib", "umap-learn", "langid", "pyyaml"], check=True)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "GPU не підключено. Runtime -> Change runtime type -> T4 GPU, потім запустіть знову."
print("Дані:", RAW)
print("Результати:", OUT)
print("Кеш ембедингів:", INTERIM)


In [ ]:
# 2. Код репозиторію і малі вхідні таблиці (вбудовані в ноутбук)
FILES = {'src/narr/__init__.py': '"""Код магістерської роботи: гібридні моделі (граф + текст) для виявлення дезінформаційних наративів."""\n\n__version__ = "0.1.0"\n', 'src/narr/eda.py': '"""S2: описова аналітика повних даних TeleNarratives.\n\nМодуль самодостатній (pandas, numpy, pyarrow, matplotlib), тому той самий код працює в Colab і в тестах.\nУсі функції приймають таблиці як DataFrame і не читають диск, крім `load_inputs`.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nKEY_MULT = 10_000_000          # ключ повідомлення = channel_id * KEY_MULT + message_id\nDATE_MIN, DATE_MAX = "2024-12-16", "2026-02-27"\n# Кількості з README і MANIFEST.json графового набору (знімок 11.03.2026)\nREADME_COUNTS = {"channels": 111, "messages": 1_052_563, "forward_edges": 35_697, "semantic_edges": 52_414,\n                 "channel_flow_edges": 214, "sem_channel_flow_edges": 1_273}\nGOLD_COLS = {"text": "text", "channel": "channel_id", "message": "message_id", "date": "date_utc",\n             "binary": "Binary label", "level1": "General label (level-1)"}\nMSG_COLS = ["channel_id", "message_id", "date_utc", "channel_username", "text", "views", "forwards", "has_media",\n            "binary_label", "is_labeled_batch", "narrative_id", "sub_narrative_id", "label_confidence"]\nTG_COLS = ["channel_id", "message_id", "is_reply", "reply_to_msg_id", "is_forwarded", "fwd_from_channel_id",\n           "fwd_from_message_id", "date_utc", "views", "forwards", "replies_count", "has_media"]\n\n_NON_WORD = re.compile(r"[^\\w\\s]+", re.UNICODE)\n\n\n# ---------------------------------------------------------------- допоміжні\n\ndef keys(df: pd.DataFrame, ch: str = "channel_id", msg: str = "message_id") -> pd.Series:\n    return df[ch].astype("int64") * KEY_MULT + df[msg].astype("int64")\n\n\ndef _flag(s: pd.Series) -> pd.Series:\n    """Булева маска з порожніми значеннями як False (працює для bool, object з None, nullable boolean)."""\n    return s.eq(True).fillna(False).astype(bool)\n\n\ndef normalize_text(s: pd.Series) -> pd.Series:\n    out = s.fillna("").astype(str).str.lower().str.replace(_NON_WORD, "", regex=True)\n    return out.str.replace(r"\\s+", " ", regex=True).str.strip()\n\n\ndef _share(a, b) -> float:\n    return float(a) / float(b) if b else float("nan")\n\n\ndef _binary_scores(y: np.ndarray, p: np.ndarray) -> dict:\n    y, p = np.asarray(y, bool), np.asarray(p, bool)\n    tp, fp = int((y & p).sum()), int((~y & p).sum())\n    fn, tn = int((y & ~p).sum()), int((~y & ~p).sum())\n\n    def f1(t, f_p, f_n):\n        d = 2 * t + f_p + f_n\n        return 2 * t / d if d else float("nan")\n\n    den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))\n    return {"n": int(len(y)), "tp": tp, "fp": fp, "fn": fn, "tn": tn,\n            "accuracy": _share(tp + tn, len(y)), "precision": _share(tp, tp + fp), "recall": _share(tp, tp + fn),\n            "f1_pos": f1(tp, fp, fn), "macro_f1": (f1(tp, fp, fn) + f1(tn, fn, fp)) / 2,\n            "mcc": (tp * tn - fp * fn) / den if den else float("nan")}\n\n\n# ---------------------------------------------------------------- читання\n\ndef load_inputs(raw_dir: Path | str, golden_path: Path | str) -> dict:\n    """Читає потрібні таблиці з data/raw (повні parquet не тримаємо в пам\'яті, якщо вони не потрібні)."""\n    import pyarrow.parquet as pq\n\n    raw_dir = Path(raw_dir)\n    mpath = next(iter(sorted(raw_dir.rglob("messages.parquet"))), None)\n    if mpath is None:\n        raise FileNotFoundError(f"не знайдено messages.parquet у {raw_dir}")\n    gd = mpath.parent\n    tg_path = raw_dir / "telegram_messages.parquet"\n\n    msgs = pq.read_table(mpath, columns=MSG_COLS).to_pandas()\n    tg = pq.read_table(tg_path, columns=TG_COLS).to_pandas()\n    gold = pd.read_csv(golden_path)\n\n    gold_ids = [int(x) for x in gold[GOLD_COLS["message"]].unique()]\n    tg_text = pq.read_table(tg_path, columns=["channel_id", "message_id", "text"],\n                            filters=[("message_id", "in", gold_ids)]).to_pandas()\n\n    fwd = pq.read_table(gd / "forward_edges.parquet",\n                        columns=["logical_src_channel_id", "logical_src_message_id",\n                                 "logical_dst_channel_id", "logical_dst_message_id"]).to_pandas()\n    sem = pq.read_table(gd / "semantic_edges.parquet",\n                        columns=["src_channel_id", "src_message_id", "dst_channel_id", "dst_message_id",\n                                 "similarity_score"]).to_pandas()\n    return {"msgs": msgs, "tg": tg, "tg_gold_text": tg_text, "gold": gold, "fwd": fwd, "sem": sem,\n            "channels": pq.read_table(gd / "channels.parquet").to_pandas(),\n            "coding": pd.read_csv(gd / "narrative_subnarrative_coding_en.csv"),\n            "paths": {"messages": str(mpath), "telegram_messages": str(tg_path)}}\n\n\n# ---------------------------------------------------------------- таблиці\n\ndef check_integrity(msgs: pd.DataFrame, tg: pd.DataFrame, date_min: str = DATE_MIN, date_max: str = DATE_MAX) -> dict:\n    km, kt = keys(msgs).to_numpy(), keys(tg).to_numpy()\n    both = np.intersect1d(km, kt)\n    lo = pd.Timestamp(date_min, tz="UTC")\n    hi = pd.Timestamp(date_max, tz="UTC") + pd.Timedelta(days=1)\n    out = {"n_msgs": int(len(msgs)), "n_tg": int(len(tg)),\n           "n_channels_msgs": int(msgs["channel_id"].nunique()), "n_channels_tg": int(tg["channel_id"].nunique()),\n           "dup_keys_msgs": int(pd.Series(km).duplicated().sum()), "dup_keys_tg": int(pd.Series(kt).duplicated().sum()),\n           "keys_both": int(len(both)), "keys_only_msgs": int(len(np.unique(km)) - len(both)),\n           "keys_only_tg": int(len(np.unique(kt)) - len(both))}\n    for name, df in (("msgs", msgs), ("tg", tg)):\n        d = pd.to_datetime(df["date_utc"], utc=True, errors="coerce")\n        out[f"unparsed_dates_{name}"] = int(d.isna().sum())\n        out[f"date_min_{name}"] = str(d.min())\n        out[f"date_max_{name}"] = str(d.max())\n        out[f"out_of_range_{name}"] = int(((d < lo) | (d >= hi)).sum())\n    return out\n\n\ndef monthly_table(msgs: pd.DataFrame) -> pd.DataFrame:\n    d = pd.to_datetime(msgs["date_utc"], utc=True)\n    lab = _flag(msgs["is_labeled_batch"])\n    t = pd.DataFrame({"month": d.dt.strftime("%Y-%m"), "n": 1, "n_labeled": lab.astype(int),\n                      "n_narrative": (lab & msgs["narrative_id"].notna()).astype(int)})\n    t = t.groupby("month", as_index=False).sum()\n    t["narrative_share_of_labeled"] = t["n_narrative"] / t["n_labeled"].where(t["n_labeled"] > 0)\n    t["partial_month"] = t["month"].isin([t["month"].min(), t["month"].max()])\n    return t\n\n\ndef per_channel(msgs: pd.DataFrame, tg: pd.DataFrame, channels: pd.DataFrame) -> pd.DataFrame:\n    lab = _flag(msgs["is_labeled_batch"])\n    g = pd.DataFrame({"channel_id": msgs["channel_id"], "n_messages": 1, "n_labeled": lab.astype(int),\n                      "n_narrative": (lab & msgs["narrative_id"].notna()).astype(int),\n                      "length": msgs["text"].fillna("").str.len()})\n    agg = g.groupby("channel_id").agg(n_messages=("n_messages", "sum"), n_labeled=("n_labeled", "sum"),\n                                      n_narrative=("n_narrative", "sum"), median_length=("length", "median")).reset_index()\n    t2 = tg.groupby("channel_id").agg(forwarded_share=("is_forwarded", "mean"), reply_share=("is_reply", "mean")).reset_index()\n    ch = channels[["channel_id", "username", "narrative_presence", "narrative_presence_status", "channel_label",\n                   "labeled_messages", "narrative_messages", "narrative_share"]].rename(columns={\n        "narrative_presence": "table_narrative_presence", "narrative_presence_status": "table_status",\n        "channel_label": "table_label", "labeled_messages": "table_labeled", "narrative_messages": "table_narrative",\n        "narrative_share": "table_share"})\n    out = ch.merge(agg, on="channel_id", how="outer").merge(t2, on="channel_id", how="left")\n    for c in ("n_messages", "n_labeled", "n_narrative"):\n        out[c] = out[c].fillna(0).astype(int)\n    out["share_recomputed"] = out["n_narrative"] / out["n_labeled"].where(out["n_labeled"] > 0)\n    out["counts_match"] = (out["n_labeled"] == out["table_labeled"]) & (out["n_narrative"] == out["table_narrative"])\n    return out.sort_values("n_messages", ascending=False).reset_index(drop=True)\n\n\ndef label_tables(msgs: pd.DataFrame, coding: pd.DataFrame) -> tuple[dict, dict[str, pd.DataFrame]]:\n    lab = _flag(msgs["is_labeled_batch"])\n    narr = msgs["narrative_id"].notna()\n    n_lab, n_narr = int(lab.sum()), int((lab & narr).sum())\n    summary = {"n_messages": int(len(msgs)), "n_labeled": n_lab, "share_labeled": _share(n_lab, len(msgs)),\n               "n_narrative_any": int(narr.sum()), "n_narrative_labeled": n_narr,\n               "narrative_not_labeled": int((narr & ~lab).sum()),\n               "narrative_share_of_labeled": _share(n_narr, n_lab),\n               "sub_without_narrative": int((msgs["sub_narrative_id"].notna() & ~narr).sum()),\n               "confidence_missing_among_labeled": int(msgs.loc[lab, "label_confidence"].isna().sum())}\n\n    # consistency of sub_narrative_id -> narrative_id with the coding file\n    sub2narr = coding.drop_duplicates("sub_narrative_id").set_index("sub_narrative_id")["narrative_id"]\n    both = msgs.loc[msgs["sub_narrative_id"].notna() & narr, ["sub_narrative_id", "narrative_id"]]\n    mapped = both["sub_narrative_id"].astype("int64").map(sub2narr)\n    summary["sub_narrative_unknown_in_coding"] = int(mapped.isna().sum())\n    summary["sub_to_narrative_match_share"] = _share(int((mapped == both["narrative_id"]).sum()), len(both))\n    summary["max_sub_narrative_id"] = int(msgs["sub_narrative_id"].max()) if msgs["sub_narrative_id"].notna().any() else None\n    summary["sub_ids_not_in_coding"] = int(msgs["sub_narrative_id"].dropna().astype("int64").map(sub2narr).isna().sum())\n\n    # confidence of labels\n    conf = pd.DataFrame({"label_confidence": msgs.loc[lab, "label_confidence"], "narrative": narr[lab]})\n    conf_t = (conf.groupby("label_confidence")["narrative"].agg(n="size", n_narrative="sum").reset_index()\n              .assign(n_not_narrative=lambda d: d["n"] - d["n_narrative"]))\n    conf_t["n_narrative"] = conf_t["n_narrative"].astype(int)\n\n    # binary_label vs labeled batch\n    bl = msgs["binary_label"].astype(object).where(msgs["binary_label"].notna(), "(порожньо)").astype(str)\n    cross = (pd.DataFrame({"binary_label": bl, "is_labeled_batch": lab, "has_narrative_id": narr})\n             .groupby(["binary_label", "is_labeled_batch", "has_narrative_id"]).size().rename("n").reset_index())\n\n    # meta-narratives of weak labels (through narrative_id, not sub_narrative_id)\n    n2m = coding.drop_duplicates("narrative_id")[["narrative_id", "level_1_id", "level_1"]]\n    summary["narrative_to_meta_unique"] = bool((coding.groupby("narrative_id")["level_1_id"].nunique() == 1).all())\n    m = msgs.loc[lab & narr, ["narrative_id"]].merge(n2m, on="narrative_id", how="left")\n    meta = (m.groupby(["level_1_id", "level_1"], dropna=False).size().rename("n_weak").reset_index()\n            .sort_values("n_weak", ascending=False))\n    meta["share_weak"] = meta["n_weak"] / meta["n_weak"].sum()\n\n    # text length\n    L = msgs["text"].fillna("").str.len()\n    groups = {"усі": pd.Series(True, index=msgs.index), "розмічені": lab, "розмічені, наративні": lab & narr,\n              "розмічені, ненаративні": lab & ~narr, "нерозмічені": ~lab}\n    rows = []\n    for name, mask in groups.items():\n        x = L[mask]\n        rows.append({"group": name, "n": int(len(x)), "mean": float(x.mean()), "p10": float(x.quantile(.1)),\n                     "median": float(x.median()), "p90": float(x.quantile(.9)), "max": int(x.max())})\n    return summary, {"label_confidence": conf_t, "binary_label_crosstab": cross, "meta_weak": meta,\n                     "length_stats": pd.DataFrame(rows)}\n\n\ndef language_tables(msgs: pd.DataFrame, gold: pd.DataFrame, lang_fn, n_unlabeled: int, seed: int,\n                    gcols: dict = GOLD_COLS) -> dict[str, pd.DataFrame]:\n    """Мова всіх розмічених + випадкової вибірки нерозмічених + золотих. lang_fn(Series) -> масив кодів."""\n    lab = _flag(msgs["is_labeled_batch"]).to_numpy()\n    rng = np.random.default_rng(seed)\n    idx_lab = np.flatnonzero(lab)\n    idx_unl = np.flatnonzero(~lab)\n    take = np.sort(rng.choice(idx_unl, size=min(n_unlabeled, len(idx_unl)), replace=False)) if len(idx_unl) else idx_unl\n    sub = msgs.iloc[np.concatenate([idx_lab, take])][["channel_id", "message_id", "text", "narrative_id"]].copy()\n    sub["is_labeled"] = np.r_[np.ones(len(idx_lab), bool), np.zeros(len(take), bool)]\n    sub["narrative"] = sub["narrative_id"].notna() & sub["is_labeled"]\n    sub["lang"] = lang_fn(sub["text"].fillna("").astype(str).str.slice(0, 300))\n\n    by_group = (sub.assign(group=np.where(sub["is_labeled"], "розмічені (усі)", "нерозмічені (вибірка)"))\n                .groupby(["group", "lang"]).size().rename("n").reset_index())\n    by_group["share"] = by_group["n"] / by_group.groupby("group")["n"].transform("sum")\n\n    wl = sub[sub["is_labeled"]]\n    x_label = (wl.groupby("lang")["narrative"].agg(n="size", n_narrative="sum").reset_index()\n               .assign(source="слабкі мітки (розмічені)"))\n\n    g = gold.copy()\n    g["lang"] = lang_fn(g[gcols["text"]].fillna("").astype(str).str.slice(0, 300))\n    gx = (g.groupby("lang")[gcols["binary"]].agg(n="size", n_narrative="sum").reset_index()\n          .assign(source="золота розмітка"))\n    x_label = pd.concat([x_label, gx], ignore_index=True)\n    x_label["n_narrative"] = x_label["n_narrative"].astype(int)\n    x_label["narrative_share"] = x_label["n_narrative"] / x_label["n"]\n\n    ch = sub.groupby(["channel_id", "lang"]).size().unstack(fill_value=0)\n    ch["n_language_sample"] = ch.sum(axis=1)\n    for c in [c for c in ch.columns if c != "n_language_sample"]:\n        ch[f"share_{c}"] = ch[c] / ch["n_language_sample"]\n    return {"language_by_group": by_group, "language_x_label": x_label, "language_by_channel": ch.reset_index()}\n\n\ndef graph_membership(msgs: pd.DataFrame, fwd: pd.DataFrame, sem: pd.DataFrame) -> pd.DataFrame:\n    km = keys(msgs).to_numpy()\n    f = np.union1d(keys(fwd, "logical_src_channel_id", "logical_src_message_id").to_numpy(),\n                   keys(fwd, "logical_dst_channel_id", "logical_dst_message_id").to_numpy())\n    s = np.union1d(keys(sem, "src_channel_id", "src_message_id").to_numpy(),\n                   keys(sem, "dst_channel_id", "dst_message_id").to_numpy())\n    in_f, in_s = np.isin(km, f), np.isin(km, s)\n    lab = _flag(msgs["is_labeled_batch"]).to_numpy()\n    narr = (lab & msgs["narrative_id"].notna().to_numpy())\n    grp = np.where(in_f & in_s, "forward і semantic", np.where(in_f, "лише forward",\n                                                              np.where(in_s, "лише semantic", "поза графом")))\n    t = (pd.DataFrame({"group": grp, "n": 1, "n_labeled": lab.astype(int), "n_narrative": narr.astype(int)})\n         .groupby("group", as_index=False).sum())\n    t["labeled_share"] = t["n_labeled"] / t["n"]\n    t["narrative_share_of_labeled"] = t["n_narrative"] / t["n_labeled"].where(t["n_labeled"] > 0)\n    return t\n\n\ndef forwards_summary(tg: pd.DataFrame, msgs: pd.DataFrame, channels: pd.DataFrame, n_forward_edges: int) -> dict:\n    fw = tg[_flag(tg["is_forwarded"])]\n    corpus = set(channels["channel_id"].astype("int64")) | set(msgs["channel_id"].astype("int64"))\n    src = fw["fwd_from_channel_id"].dropna().astype("int64")\n    in_corpus_channel = src.isin(corpus)\n    ok = fw["fwd_from_channel_id"].notna() & fw["fwd_from_message_id"].notna()\n    ofk = (fw.loc[ok, "fwd_from_channel_id"].astype("int64") * KEY_MULT + fw.loc[ok, "fwd_from_message_id"].astype("int64"))\n    orig_present = ofk.isin(set(keys(tg)))\n    return {"n_forwarded": int(len(fw)), "forwarded_share": _share(len(fw), len(tg)),\n            "n_unique_source_channels": int(src.nunique()),\n            "n_forwarded_from_corpus_channel": int(in_corpus_channel.sum()),\n            "n_forwarded_with_original_in_corpus": int(orig_present.sum()),\n            "n_forward_edges_file": int(n_forward_edges),\n            "n_replies": int(_flag(tg["is_reply"]).sum()), "reply_share": _share(int(_flag(tg["is_reply"]).sum()), len(tg))}\n\n\ndef _gold_merge(gold, msgs, gcols):\n    g = gold.copy()\n    g["_key"] = keys(g, gcols["channel"], gcols["message"])\n    m = msgs[["channel_id", "message_id", "text", "is_labeled_batch", "narrative_id", "label_confidence"]].copy()\n    m["_key"] = keys(m)\n    return g.merge(m[["_key", "text", "is_labeled_batch", "narrative_id", "label_confidence"]].rename(\n        columns={"text": "text_graph"}), on="_key", how="left")\n\n\ndef gold_overlap(gold: pd.DataFrame, msgs: pd.DataFrame, coding: pd.DataFrame, tg_gold_text: pd.DataFrame,\n                 fwd: pd.DataFrame, sem: pd.DataFrame, gcols: dict = GOLD_COLS) -> tuple[dict, dict[str, pd.DataFrame]]:\n    m = _gold_merge(gold, msgs, gcols)\n    present = m["text_graph"].notna()\n    lab = _flag(m["is_labeled_batch"]) & present\n    y = m[gcols["binary"]].astype(bool)\n    summary = {"n_gold": int(len(m)), "n_gold_in_graph_messages": int(present.sum()),\n               "n_gold_labeled_weak": int(lab.sum()),\n               "n_gold_present_not_labeled": int((present & ~lab).sum()),\n               "gold_narrative_share_present_not_labeled": _share(int((y & present & ~lab).sum()), int((present & ~lab).sum()))}\n    km_gold = keys(gold, gcols["channel"], gcols["message"])\n    f = np.union1d(keys(fwd, "logical_src_channel_id", "logical_src_message_id").to_numpy(),\n                   keys(fwd, "logical_dst_channel_id", "logical_dst_message_id").to_numpy())\n    s = np.union1d(keys(sem, "src_channel_id", "src_message_id").to_numpy(),\n                   keys(sem, "dst_channel_id", "dst_message_id").to_numpy())\n    summary["n_gold_in_forward_edges"] = int(np.isin(km_gold.to_numpy(), f).sum())\n    summary["n_gold_in_semantic_edges"] = int(np.isin(km_gold.to_numpy(), s).sum())\n\n    tables: dict[str, pd.DataFrame] = {}\n    if lab.any():\n        weak = m.loc[lab, "narrative_id"].notna().to_numpy()\n        sc = _binary_scores(y[lab].to_numpy(), weak)\n        summary["weak_vs_gold"] = sc\n        tables["weak_vs_gold_confusion"] = pd.DataFrame(\n            [[sc["tn"], sc["fp"]], [sc["fn"], sc["tp"]]], index=["золота: ні", "золота: так"],\n            columns=["слабка: ні", "слабка: так"]).reset_index().rename(columns={"index": ""})\n        # мета-наративи: назви рівня 1 у золотому наборі проти слабких міток\n        n2m = coding.drop_duplicates("narrative_id").set_index("narrative_id")["level_1"]\n        both = lab & y & m["narrative_id"].notna()\n        a = m.loc[both, gcols["level1"]].astype(str)\n        b = m.loc[both, "narrative_id"].astype("int64").map(n2m).astype(str)\n        summary["meta_pairs_both_narrative"] = int(both.sum())\n        summary["meta_match_share"] = _share(int((a.to_numpy() == b.to_numpy()).sum()), int(both.sum()))\n        summary["gold_meta_names_not_in_coding"] = int((~a.isin(set(coding["level_1"]))).sum())\n        tables["weak_vs_gold_meta"] = pd.crosstab(a.rename("золотий мета-наратив"), b.rename("слабкий мета-наратив")).reset_index()\n    # варіанти тексту\n    tt = tg_gold_text.copy()\n    tt["_key"] = keys(tt)\n    mm = m.merge(tt[["_key", "text"]].rename(columns={"text": "text_tg"}), on="_key", how="left")\n    gt = mm[gcols["text"]].fillna("").astype(str)\n    gn = normalize_text(gt)\n    rows = []\n    for name, col in (("messages.parquet (сирий)", "text_graph"), ("telegram_messages.parquet (нормалізований)", "text_tg")):\n        has = mm[col].notna()\n        other = mm.loc[has, col].astype(str)\n        on = normalize_text(other)\n        gnn = gn[has]\n        rows.append({"варіант тексту": name, "n_знайдено": int(has.sum()),\n                     "точно збігається": int((gt[has].to_numpy() == other.to_numpy()).sum()),\n                     "збігається після нормалізації": int((gnn.to_numpy() == on.to_numpy()).sum()),\n                     "збігаються перші 60 символів після нормалізації": int((gnn.str[:60].to_numpy() == on.str[:60].to_numpy()).sum()),\n                     "медіана довжини": float(other.str.len().median())})\n    summary["gold_text_median_length"] = float(gt.str.len().median())\n    tables["text_variants"] = pd.DataFrame(rows)\n    # копії золотих текстів у корпусі (для виключення в P1)\n    norm_all = normalize_text(msgs["text"])\n    gold_norm = set(gn[gn != ""])\n    is_copy = norm_all.isin(gold_norm).to_numpy()\n    not_self = ~keys(msgs).isin(set(km_gold)).to_numpy()\n    summary["n_corpus_copies_of_gold_text"] = int((is_copy & not_self).sum())\n    gcopy = gn.isin(set(norm_all[is_copy & not_self]))\n    summary["n_gold_with_corpus_copy"] = int(gcopy.sum())\n    dup = norm_all.duplicated(keep=False)\n    labd = _flag(msgs["is_labeled_batch"])\n    summary["duplicates"] = {"n_rows_in_duplicate_groups": int(dup.sum()), "share_all": float(dup.mean()),\n                             "share_labeled": float(dup[labd.to_numpy()].mean()), "n_unique_texts": int(norm_all.nunique())}\n    return summary, tables\n\n\n# ---------------------------------------------------------------- запуск\n\ndef run_all(inp: dict, lang_fn, lang_sample_unlabeled: int = 150_000, seed: int = 42,\n            gcols: dict = GOLD_COLS) -> tuple[dict, dict, list]:\n    msgs, tg, channels, coding = inp["msgs"], inp["tg"], inp["channels"], inp["coding"]\n    gold = inp["gold"]\n    tables: dict[str, pd.DataFrame] = {}\n    summary: dict = {"readme_counts": README_COUNTS}\n    summary["integrity"] = check_integrity(msgs, tg)\n    tables["monthly"] = monthly_table(msgs)\n    tables["per_channel"] = per_channel(msgs, tg, channels)\n    s, t = label_tables(msgs, coding)\n    summary["labels"] = s\n    tables.update(t)\n    tables.update(language_tables(msgs, gold, lang_fn, lang_sample_unlabeled, seed, gcols))\n    tables["graph_membership"] = graph_membership(msgs, inp["fwd"], inp["sem"])\n    summary["forwards"] = forwards_summary(tg, msgs, channels, len(inp["fwd"]))\n    gs, gt = gold_overlap(gold, msgs, coding, inp["tg_gold_text"], inp["fwd"], inp["sem"], gcols)\n    summary["gold"] = gs\n    tables.update(gt)\n    gb = gold[gcols["binary"]].astype(bool)\n    summary["gold_narrative_share"] = float(gb.mean())\n    gm = gold.loc[gb, gcols["level1"]].astype(str).value_counts()\n    tables["gold_meta_counts"] = pd.DataFrame({"level_1": gm.index, "n_gold": gm.to_numpy(),\n                                               "share_gold": gm.to_numpy() / gm.sum()})\n    summary["edges"] = {"forward_edges": int(len(inp["fwd"])), "semantic_edges": int(len(inp["sem"])),\n                        "semantic_min_similarity": float(inp["sem"]["similarity_score"].min())}\n    return tables, summary, run_checks(tables, summary)\n\n\ndef run_checks(tables: dict, summary: dict) -> list[dict]:\n    """hard=True: інваріанти (дублікати ключів, непарсовані дати), їх порушення ламає наступні етапи.\n    hard=False: очікування з README і плану, розбіжність не блокує етап, а потрапляє у звіт як знахідка."""\n    it, lb, rc = summary["integrity"], summary["labels"], summary["readme_counts"]\n    pc = tables["per_channel"]\n    pc_msgs = pc[pc["n_messages"] > 0]\n    checks = [\n        (True, "Немає дублікатів (channel_id, message_id) у messages", it["dup_keys_msgs"] == 0, f"дублікатів {it[\'dup_keys_msgs\']}"),\n        (True, "Немає дублікатів (channel_id, message_id) у telegram_messages", it["dup_keys_tg"] == 0, f"дублікатів {it[\'dup_keys_tg\']}"),\n        (True, "Усі дати розпарсені", it["unparsed_dates_msgs"] == 0 and it["unparsed_dates_tg"] == 0,\n         f"непарсованих {it[\'unparsed_dates_msgs\']} і {it[\'unparsed_dates_tg\']}"),\n        (False, f"Дати в межах {DATE_MIN} .. {DATE_MAX}", it["out_of_range_msgs"] == 0 and it["out_of_range_tg"] == 0,\n         f"поза межами {it[\'out_of_range_msgs\']} (messages), {it[\'out_of_range_tg\']} (telegram_messages)"),\n        (False, "Наративні повідомлення входять до розмічених", lb["narrative_not_labeled"] == 0, f"поза розміченими {lb[\'narrative_not_labeled\']}"),\n        (False, "sub_narrative_id у мітках відомі файлу кодування", lb["sub_ids_not_in_coding"] == 0, f"невідомих {lb[\'sub_ids_not_in_coding\']}"),\n        (False, "Кожен sub_narrative_id веде до того самого narrative_id, що й у мітці", lb["sub_to_narrative_match_share"] == 1.0,\n         f"збіг {lb[\'sub_to_narrative_match_share\']:.4f}"),\n        (False, "Кількість повідомлень збігається з README", it["n_msgs"] == rc["messages"], f"{it[\'n_msgs\']} проти {rc[\'messages\']}"),\n        (False, "Кількість каналів у таблиці каналів збігається з README", len(pc) == rc["channels"], f"{len(pc)} проти {rc[\'channels\']}"),\n        (False, "Кількості ребер збігаються з README",\n         summary["edges"]["forward_edges"] == rc["forward_edges"] and summary["edges"]["semantic_edges"] == rc["semantic_edges"],\n         f"forward {summary[\'edges\'][\'forward_edges\']}, semantic {summary[\'edges\'][\'semantic_edges\']}"),\n        (False, "Частка наративних серед розмічених близька до золотої (різниця до 0,10)",\n         abs(lb["narrative_share_of_labeled"] - summary["gold_narrative_share"]) <= 0.10,\n         f"слабкі {lb[\'narrative_share_of_labeled\']:.3f}, золота {summary[\'gold_narrative_share\']:.3f}"),\n        (False, "Мітки каналів у таблиці збігаються з перерахунком за повідомленнями", bool(pc_msgs["counts_match"].all()),\n         f"збігається {int(pc_msgs[\'counts_match\'].sum())} з {len(pc_msgs)} каналів"),\n        (False, "Усі золоті повідомлення знайдені в messages", summary["gold"]["n_gold_in_graph_messages"] == summary["gold"]["n_gold"],\n         f"{summary[\'gold\'][\'n_gold_in_graph_messages\']} з {summary[\'gold\'][\'n_gold\']}"),\n    ]\n    return [{"hard": h, "check": n, "passed": bool(ok), "detail": d} for h, n, ok, d in checks]\n\n\n# ---------------------------------------------------------------- збереження\n\ndef _default(o):\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, (np.floating,)):\n        return None if np.isnan(o) else float(o)\n    if isinstance(o, (np.bool_,)):\n        return bool(o)\n    if isinstance(o, (pd.Timestamp,)):\n        return str(o)\n    raise TypeError(type(o))\n\n\ndef save_all(tables: dict, summary: dict, checks: list, out_dir: Path | str) -> list[Path]:\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    paths = []\n    for name, df in tables.items():\n        p = out_dir / f"{name}.csv"\n        df.to_csv(p, index=False)\n        paths.append(p)\n    (out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=_default), encoding="utf-8")\n    (out_dir / "checks.json").write_text(json.dumps(checks, ensure_ascii=False, indent=2), encoding="utf-8")\n    return paths + [out_dir / "summary.json", out_dir / "checks.json"]\n\n\n# ---------------------------------------------------------------- графіки\n\ndef make_figures(tables: dict, summary: dict, out_dir: Path | str) -> list[Path]:\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150, "axes.spines.top": False, "axes.spines.right": False,\n                         "font.size": 9})\n    paths = []\n\n    def save(fig, name):\n        p = out_dir / name\n        fig.savefig(p, bbox_inches="tight")\n        plt.close(fig)\n        paths.append(p)\n\n    # 1. Повідомлення за місяцями\n    mo = tables["monthly"]\n    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))\n    colors = np.where(mo["partial_month"], "#b8b8b8", "#3b6fb6")\n    ax[0].bar(mo["month"], mo["n"] / 1000, color=colors)\n    ax[0].set_ylabel("повідомлень, тис.")\n    ax[0].set_title("Усі повідомлення за місяцями (сірі: неповні місяці)")\n    ax[1].bar(mo["month"], mo["n_labeled"] / 1000, color="#8aa9d6", label="розмічені")\n    ax[1].bar(mo["month"], mo["n_narrative"] / 1000, color="#c0392b", label="з наративом")\n    ax[1].set_ylabel("повідомлень, тис.")\n    ax[1].set_title("Слабкі мітки за місяцями")\n    ax[1].legend(frameon=False)\n    for a in ax:\n        a.tick_params(axis="x", rotation=70)\n    save(fig, "01_monthly.png")\n\n    # 2. Канали: обсяг і частка наративних\n    pc = tables["per_channel"]\n    top = pc[pc["n_messages"] > 0].head(25).iloc[::-1]\n    fig, ax = plt.subplots(1, 2, figsize=(11, 6), sharey=True)\n    ax[0].barh(top["username"], top["n_messages"] / 1000, color="#3b6fb6")\n    ax[0].set_xlabel("повідомлень, тис.")\n    ax[0].set_title("25 найбільших каналів")\n    ax[1].barh(top["username"], top["share_recomputed"].fillna(0), color="#c0392b")\n    ax[1].set_xlabel("частка наративних серед розмічених")\n    ax[1].set_xlim(0, 1)\n    ax[1].set_title("Слабкі мітки")\n    save(fig, "02_channels.png")\n\n    # 3. Мова і частка наративних\n    lx = tables["language_x_label"]\n    fig, ax = plt.subplots(figsize=(6.5, 3.8))\n    langs = sorted(lx["lang"].unique())\n    w = 0.38\n    for i, (src, col) in enumerate((("слабкі мітки (розмічені)", "#8aa9d6"), ("золота розмітка", "#c0392b"))):\n        d = lx[lx["source"] == src].set_index("lang").reindex(langs)\n        xs = np.arange(len(langs)) + (i - 0.5) * w\n        ax.bar(xs, d["narrative_share"].fillna(0), w, label=src, color=col)\n        for x, (_, r) in zip(xs, d.iterrows()):\n            if pd.notna(r["n"]):\n                ax.text(x, (r["narrative_share"] if pd.notna(r["narrative_share"]) else 0) + 0.01, f"n={int(r[\'n\'])}",\n                        ha="center", fontsize=7)\n    ax.set_xticks(np.arange(len(langs)))\n    ax.set_xticklabels(langs)\n    ax.set_ylabel("частка наративних")\n    ax.set_ylim(0, 1.1)\n    ax.set_title("Мова і частка наративних повідомлень")\n    ax.legend(frameon=False)\n    save(fig, "03_language.png")\n\n    # 4. Мета-наративи: слабкі мітки проти золотої розмітки\n    meta = tables["meta_weak"].copy()\n    gold_meta = tables.get("gold_meta_counts")\n    fig, ax = plt.subplots(figsize=(9, 4.2))\n    meta = meta.sort_values("n_weak")\n    ax.barh(meta["level_1"].astype(str).str.slice(0, 48), meta["share_weak"], color="#8aa9d6", label="слабкі мітки")\n    if gold_meta is not None:\n        gm = gold_meta.set_index("level_1")["share_gold"]\n        ax.scatter(meta["level_1"].map(gm).fillna(0), range(len(meta)), color="#c0392b", zorder=3, label="золота розмітка")\n    ax.set_xlabel("частка серед наративних")\n    ax.set_title("Розподіл мета-наративів")\n    ax.legend(frameon=False)\n    save(fig, "04_meta.png")\n\n    # 5. Довжина тексту\n    ls = tables["length_stats"].set_index("group")\n    fig, ax = plt.subplots(figsize=(6.5, 3.6))\n    cols = ["розмічені, ненаративні", "розмічені, наративні", "нерозмічені"]\n    ax.bar(range(len(cols)), [ls.loc[c, "median"] for c in cols], color=["#8aa9d6", "#c0392b", "#b8b8b8"])\n    ax.errorbar(range(len(cols)), [ls.loc[c, "median"] for c in cols],\n                yerr=[[ls.loc[c, "median"] - ls.loc[c, "p10"] for c in cols], [ls.loc[c, "p90"] - ls.loc[c, "median"] for c in cols]],\n                fmt="none", color="k", capsize=4)\n    ax.set_xticks(range(len(cols)))\n    ax.set_xticklabels(cols, fontsize=8)\n    ax.set_ylabel("символів (медіана, діапазон p10-p90)")\n    ax.set_title("Довжина тексту")\n    save(fig, "05_length.png")\n\n    # 6. Упевненість міток і слабкі мітки проти золотих\n    cf = tables["label_confidence"]\n    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))\n    ax[0].bar(cf["label_confidence"].astype(str), cf["n_not_narrative"], label="без наративу", color="#8aa9d6")\n    ax[0].bar(cf["label_confidence"].astype(str), cf["n_narrative"], bottom=cf["n_not_narrative"], label="з наративом", color="#c0392b")\n    ax[0].set_xlabel("label_confidence")\n    ax[0].set_ylabel("розмічених повідомлень")\n    ax[0].set_title("Упевненість слабких міток")\n    ax[0].tick_params(axis="x", rotation=60)\n    ax[0].legend(frameon=False)\n    cm = tables.get("weak_vs_gold_confusion")\n    if cm is not None:\n        arr = cm.iloc[:, 1:].to_numpy(dtype=float)\n        ax[1].imshow(arr, cmap="Blues")\n        for i in range(2):\n            for j in range(2):\n                ax[1].text(j, i, f"{int(arr[i, j])}", ha="center", va="center", color="k")\n        ax[1].set_xticks([0, 1])\n        ax[1].set_xticklabels(["слабка: ні", "слабка: так"])\n        ax[1].set_yticks([0, 1])\n        ax[1].set_yticklabels(["золота: ні", "золота: так"])\n        ax[1].set_title("Слабкі мітки проти золотих (де є обидві)")\n    save(fig, "06_confidence_and_agreement.png")\n    return paths\n', 'src/narr/language.py': '"""Мова повідомлення (uk / ru / en) через langid. Евристика: на коротких текстах можливі помилки."""\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef detect_language(texts, langs: tuple[str, ...] = ("uk", "ru", "en")) -> np.ndarray:\n    import langid\n\n    langid.set_languages(list(langs))\n    return np.array([langid.classify(str(t))[0] for t in texts])\n\n\ndef _detect_chunk(texts: list[str]) -> list[str]:\n    return list(detect_language(texts))\n\n\ndef detect_language_parallel(texts, n_jobs: int = 2, min_per_job: int = 2000) -> np.ndarray:\n    """Те саме, що detect_language, але частинами в кількох процесах. Порядок збережено.\n    Для малих вхідних масивів або n_jobs <= 1 працює послідовно."""\n    texts = [str(t) for t in texts]\n    if n_jobs <= 1 or len(texts) < n_jobs * min_per_job:\n        return detect_language(texts)\n    from multiprocessing import get_context\n\n    chunks = [c.tolist() for c in np.array_split(np.array(texts, dtype=object), n_jobs * 8) if len(c)]\n    with get_context("fork").Pool(n_jobs) as pool:\n        parts = pool.map(_detect_chunk, chunks)\n    return np.array([x for part in parts for x in part])\n', 'src/narr/io.py': '"""Конфігурація, завантаження даних, валідація схем, службова інформація про запуск."""\nfrom __future__ import annotations\n\nimport datetime as dt\nimport json\nimport platform\nimport subprocess\nfrom pathlib import Path\n\nimport pandas as pd\nimport yaml\n\nROOT = Path(__file__).resolve().parents[2]\nCONFIG_PATH = ROOT / "configs" / "base.yaml"\n\n\ndef load_config(path: Path | str | None = None) -> dict:\n    with open(path or CONFIG_PATH, encoding="utf-8") as f:\n        return yaml.safe_load(f)\n\n\ndef path_of(cfg: dict, key: str) -> Path:\n    return ROOT / cfg["paths"][key]\n\n\ndef data_available(cfg: dict | None = None) -> bool:\n    cfg = cfg or load_config()\n    return path_of(cfg, "golden").exists()\n\n\ndef load_golden(cfg: dict | None = None) -> pd.DataFrame:\n    cfg = cfg or load_config()\n    df = pd.read_csv(path_of(cfg, "golden"))\n    df[cfg["columns"]["date"]] = pd.to_datetime(df[cfg["columns"]["date"]], utc=True)\n    return df\n\n\ndef load_shared(cfg: dict | None = None) -> pd.DataFrame:\n    cfg = cfg or load_config()\n    return pd.read_csv(path_of(cfg, "shared"))\n\n\ndef validate_golden(df: pd.DataFrame, cfg: dict | None = None) -> list[str]:\n    """Повертає список проблем. Порожній список означає, що схема і вміст відповідають очікуванням."""\n    cfg = cfg or load_config()\n    col, exp = cfg["columns"], cfg["expected"]\n    no_narr = col["no_narrative_label"]\n    problems: list[str] = []\n\n    required = [col["text"], col["channel"], col["message"], col["date"], col["forwarded"],\n                col["binary"], col["general"], col["level1"], *col["reviewers"], *col["reviewers_level1"]]\n    missing = [c for c in required if c not in df.columns]\n    if missing:\n        return [f"відсутні колонки: {missing}"]\n\n    if len(df) != exp["n_messages"]:\n        problems.append(f"рядків {len(df)}, очікувалось {exp[\'n_messages\']}")\n    n_dup = int(df.duplicated([col["channel"], col["message"]]).sum())\n    if n_dup:\n        problems.append(f"дублікатів (channel_id, message_id): {n_dup}")\n    n_pos = int(df[col["binary"]].sum())\n    if n_pos != exp["n_narrative"]:\n        problems.append(f"наративних {n_pos}, очікувалось {exp[\'n_narrative\']}")\n    if int(df[col["channel"]].nunique()) != exp["n_channels_in_gold"]:\n        problems.append(f"каналів у вибірці {df[col[\'channel\']].nunique()}, очікувалось {exp[\'n_channels_in_gold\']}")\n    if df[col["text"]].isna().any() or (df[col["text"]].astype(str).str.strip() == "").any():\n        problems.append("є порожні тексти")\n    if df[col["date"]].isna().any():\n        problems.append("є непарсовані дати")\n    if ((df[col["general"]] != no_narr) != df[col["binary"]]).any():\n        problems.append("Binary label не узгоджена з General label")\n    if ((df[col["level1"]] != no_narr) != df[col["binary"]]).any():\n        problems.append("Binary label не узгоджена з General label (level-1)")\n    return problems\n\n\ndef _git(*args: str) -> str | None:\n    try:\n        out = subprocess.run(["git", "-C", str(ROOT), *args], capture_output=True, text=True, check=True)\n        return out.stdout.strip()\n    except Exception:\n        return None\n\n\ndef write_run_json(out_dir: Path | str, script: str, params: dict) -> Path:\n    """Пише run.json: дата, git-хеш, параметри, версії бібліотек."""\n    import numpy\n    import sklearn\n\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    info = {\n        "script": script,\n        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),\n        "git_commit": _git("rev-parse", "HEAD"),\n        "git_dirty": bool(_git("status", "--porcelain", "--untracked-files=no", "--", "src", "scripts", "configs")),  # зміни коду відносно коміту\n        "params": params,\n        "versions": {\n            "python": platform.python_version(),\n            "numpy": numpy.__version__,\n            "pandas": pd.__version__,\n            "scikit-learn": sklearn.__version__,\n        },\n    }\n    path = out_dir / "run.json"\n    path.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")\n    return path\n', 'src/narr/viz.py': '"""Графіки: 150 dpi, підписи українською."""\nfrom __future__ import annotations\n\nimport textwrap\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt  # noqa: E402\nimport numpy as np  # noqa: E402\nimport pandas as pd  # noqa: E402\n\nDPI = 150\nC_POS, C_NEG, C_ACC = "#c0392b", "#7f8c8d", "#2c3e50"\n\n\ndef _style():\n    plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,\n                         "figure.dpi": DPI, "savefig.dpi": DPI, "font.family": "DejaVu Sans"})\n\n\ndef wrap(s: str, w: int = 34) -> str:\n    return "\\n".join(textwrap.wrap(str(s), w))\n\n\ndef fig_gold_overview(monthly: pd.DataFrame, meta: pd.DataFrame, out: Path) -> None:\n    """monthly: колонки month, n_narrative, n_other. meta: колонки label, n."""\n    _style()\n    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.1, 1]})\n    x = np.arange(len(monthly))\n    ax[0].bar(x, monthly["n_other"], color=C_NEG, label="без наративу")\n    ax[0].bar(x, monthly["n_narrative"], bottom=monthly["n_other"], color=C_POS, label="наративні")\n    ax[0].set_xticks(x, monthly["month"], rotation=60, ha="right")\n    ax[0].set_ylabel("повідомлень")\n    ax[0].set_title("Золота вибірка за місяцями")\n    ax[0].set_ylim(0, float((monthly["n_other"] + monthly["n_narrative"]).max()) * 1.25)\n    ax[0].legend(frameon=False, ncol=2, loc="upper left")\n    m = meta.sort_values("n")\n    ax[1].barh([wrap(s) for s in m["label"]], m["n"], color=C_ACC)\n    for i, v in enumerate(m["n"]):\n        ax[1].text(v + 0.5, i, str(v), va="center")\n    ax[1].set_xlabel("наративних повідомлень")\n    ax[1].set_title("Мета-наративи (рівень 1)")\n    fig.tight_layout()\n    out.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(out)\n    plt.close(fig)\n\n\ndef fig_metrics_ci(table: pd.DataFrame, out: Path, title: str) -> None:\n    """table: колонки setting, model, metric, point, ci_lo, ci_hi (long). Три панелі: macro_f1, mcc, pr_auc."""\n    _style()\n    metrics = [("macro_f1", "macro-F1"), ("mcc", "MCC"), ("pr_auc", "PR-AUC")]\n    models = list(dict.fromkeys(table["model"]))\n    settings = list(dict.fromkeys(table["setting"]))\n    colors = {"random_cv": C_ACC, "channel_cv": C_POS}\n    names = {"random_cv": "випадкова CV", "channel_cv": "CV за каналами"}\n    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)\n    for ax, (key, label) in zip(axes, metrics):\n        for j, s in enumerate(settings):\n            sub = table[(table["metric"] == key) & (table["setting"] == s)].set_index("model").reindex(models)\n            y = np.arange(len(models)) + (j - (len(settings) - 1) / 2) * 0.28\n            ax.errorbar(sub["point"], y, xerr=[sub["point"] - sub["ci_lo"], sub["ci_hi"] - sub["point"]],\n                        fmt="o", color=colors.get(s, "k"), capsize=3, label=names.get(s, s))\n        ax.set_yticks(np.arange(len(models)), models)\n        ax.set_xlabel(label)\n        ax.grid(axis="x", alpha=0.3)\n        ax.invert_yaxis()\n    axes[1].legend(frameon=False, loc="center")\n    fig.suptitle(title)\n    fig.tight_layout()\n    out.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(out)\n    plt.close(fig)\n', 'src/narr/semantic.py': '"""S4: семантичний шар. Ембединги bge-m3, семантичні ребра, порівняння з ребрами набору, копії золотих, досяжність.\n\nМодуль не читає диск (крім `load_inputs`) і не імпортує torch: кодувальник передається як функція `encode(list[str]) -> ndarray`,\nтому всі обчислення тестуються на малих синтетичних даних. Ключ повідомлення = channel_id * 10^7 + message_id (як в eda.py).\nУсі косинуси рахуються на float32-векторах одиничної довжини (скалярний добуток).\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.stats import pearsonr, spearmanr\n\nfrom narr.eda import KEY_MULT, keys, normalize_text\n\nMODEL_NAME = "BAAI/bge-m3"\nTHRESHOLDS = (0.95, 0.975, 0.985, 0.99)\n\n\n# ---------------------------------------------------------------- набір для ембедингів\n\ndef load_inputs(raw_dir: Path | str, golden_path: Path | str) -> dict:\n    """Читає лише те, що потрібно S4. Тексти беремо з telegram_messages.parquet (звідти золоті, S2)."""\n    import pyarrow.parquet as pq\n\n    raw_dir = Path(raw_dir)\n    mpath = next(iter(sorted(raw_dir.rglob("messages.parquet"))), None)\n    if mpath is None:\n        raise FileNotFoundError(f"не знайдено messages.parquet у {raw_dir}")\n    gd = mpath.parent\n    msgs = pq.read_table(mpath, columns=["channel_id", "message_id", "text", "is_labeled_batch", "narrative_id"]).to_pandas()\n    tg = pq.read_table(raw_dir / "telegram_messages.parquet", columns=["channel_id", "message_id", "text"]).to_pandas()\n    fwd = pq.read_table(gd / "forward_edges.parquet", columns=["logical_src_channel_id", "logical_src_message_id",\n                                                              "logical_dst_channel_id", "logical_dst_message_id"]).to_pandas()\n    sem = pq.read_table(gd / "semantic_edges.parquet", columns=["src_channel_id", "src_message_id", "dst_channel_id",\n                                                               "dst_message_id", "similarity_score"]).to_pandas()\n    return {"msgs": msgs, "tg": tg, "gold": pd.read_csv(golden_path), "fwd": fwd, "sem": sem,\n            "coding": pd.read_csv(gd / "narrative_subnarrative_coding_en.csv")}\n\n\ndef build_embedding_set(msgs: pd.DataFrame, tg: pd.DataFrame, gold: pd.DataFrame, sem: pd.DataFrame,\n                        fwd: pd.DataFrame | None = None) -> tuple[pd.DataFrame, dict]:\n    """Що ембедимо: усі золоті, уся розмічена партія, усі кінці семантичних ребер набору.\n\n    Текст: нормалізований із telegram_messages (як у золотих); якщо ключа там немає, нормалізуємо сирий текст із messages.\n    Порожні після нормалізації тексти відкидаємо (їхні ембединги однакові і дали б фальшиві ребра), кількість повертаємо.\n    Колонки: key, channel_id, message_id, text, in_gold, in_labeled, in_sem, in_fwd, narrative_id."""\n    m = msgs[[c for c in ("channel_id", "message_id", "is_labeled_batch", "narrative_id") if c in msgs.columns]].assign(key=keys(msgs))\n    t = pd.Series(tg["text"].to_numpy(), index=keys(tg).to_numpy()).loc[lambda x: ~x.index.duplicated()]\n    g = gold.assign(key=keys(gold))\n    sk = pd.concat([keys(sem, "src_channel_id", "src_message_id"), keys(sem, "dst_channel_id", "dst_message_id")])\n    fk = (pd.concat([keys(fwd, "logical_src_channel_id", "logical_src_message_id"),\n                     keys(fwd, "logical_dst_channel_id", "logical_dst_message_id")]) if fwd is not None else pd.Series([], dtype="int64"))\n    lab = m.loc[m["is_labeled_batch"].eq(True), "key"]\n    allkeys = pd.Index(pd.concat([g["key"], lab, sk]).unique())\n    base = pd.DataFrame({"key": allkeys.to_numpy()})\n    base["channel_id"] = base["key"] // KEY_MULT\n    base["message_id"] = base["key"] % KEY_MULT\n    mi = m.drop_duplicates("key").set_index("key")\n    text = base["key"].map(t)\n    missing_tg = text.isna()\n    if missing_tg.any():\n        raw_all = pd.Series(msgs["text"].to_numpy(), index=keys(msgs).to_numpy())\n        raw = base.loc[missing_tg, "key"].map(raw_all.loc[lambda x: ~x.index.duplicated()])\n        text.loc[missing_tg] = normalize_text(raw.fillna("")).to_numpy()\n    gt = g.drop_duplicates("key").set_index("key")["text"]\n    still = text.isna() | (text.astype(str).str.strip() == "")\n    from_gold = base["key"].isin(gt.index) & still\n    text.loc[from_gold] = base.loc[from_gold, "key"].map(gt)\n    base["text"] = text.fillna("").astype(str)\n    base["in_gold"] = base["key"].isin(set(g["key"]))\n    base["in_labeled"] = base["key"].isin(set(lab))\n    base["in_sem"] = base["key"].isin(set(sk))\n    base["in_fwd"] = base["key"].isin(set(fk))\n    nar = base["key"].map(mi["narrative_id"]) if "narrative_id" in mi else pd.Series(np.nan, index=base.index)\n    base["narrative_id"] = nar\n    empty = base["text"].str.strip() == ""\n    info = {"n_candidates": int(len(base)), "n_empty_dropped": int(empty.sum()), "n_text_from_raw_fallback": int(missing_tg.sum()),\n            "n_gold": int(base["in_gold"].sum()), "n_labeled": int(base["in_labeled"].sum()), "n_sem_endpoints": int(base["in_sem"].sum()),\n            "n_gold_empty": int((empty & base["in_gold"]).sum())}\n    base = base[~empty].reset_index(drop=True)\n    info["n_embedded"] = int(len(base))\n    return base, info\n\n\ndef embed_sorted(texts: list[str], encode, batch_size: int = 128, progress=None) -> np.ndarray:\n    """Кодує в порядку зростання довжини (менше доповнень), повертає в початковому порядку. Рядки нормуються до одиничної довжини.\n    progress(done, total) викликається після кожної 50-ї партії."""\n    order = np.argsort([len(t) for t in texts], kind="stable")\n    out = None\n    for bi, s in enumerate(range(0, len(texts), batch_size)):\n        if progress is not None and bi % 50 == 0:\n            progress(s, len(texts))\n        idx = order[s:s + batch_size]\n        v = np.asarray(encode([texts[i] for i in idx]), dtype=np.float32)\n        if out is None:\n            out = np.zeros((len(texts), v.shape[1]), dtype=np.float32)\n        out[idx] = v\n    if out is None:\n        return np.zeros((0, 0), dtype=np.float32)\n    n = np.linalg.norm(out, axis=1, keepdims=True)\n    return out / np.maximum(n, 1e-12)\n\n\ndef hash_encoder(dim: int = 256):\n    """Детермінований кодувальник на хешованих символьних тригрaмах: лише для тестів і сухого запуску без GPU."""\n    import zlib\n\n    def enc(batch: list[str]) -> np.ndarray:\n        out = np.zeros((len(batch), dim), dtype=np.float32)\n        for r, t in enumerate(batch):\n            t = f"  {t}  "\n            for i in range(len(t) - 2):\n                out[r, zlib.crc32(t[i:i + 3].encode("utf-8")) % dim] += 1.0\n        return out\n\n    return enc\n\n\ndef quantize_fp16(emb: np.ndarray) -> np.ndarray:\n    """Те, що буде збережено (float16), повертаємо назад у float32 і знову нормуємо: аналіз завжди йде на тих самих числах,\n    що лежать на диску, тож повторний запуск з кешу дає той самий результат."""\n    q = emb.astype(np.float16).astype(np.float32)\n    return q / np.maximum(np.linalg.norm(q, axis=1, keepdims=True), 1e-12)\n\n\n# ---------------------------------------------------------------- сусіди\n\ndef knn_pairs(emb: np.ndarray, k: int = 80, threshold: float = 0.975, ef: int = 300, M: int = 32, seed: int = 42,\n              report_thresholds: tuple = ()) -> tuple[pd.DataFrame, dict]:\n    """Наближені k найближчих сусідів (hnswlib, внутрішній добуток = косинус для одиничних векторів).\n    Повертає унікальні неспрямовані пари i < j з cos >= threshold і лічильники насичення: вузол, у якого всі k сусідів вище порога,\n    міг мати більше сусідів, ніж ми знайшли. Лічильники дано для threshold і для кожного з report_thresholds."""\n    import hnswlib\n\n    n, d = emb.shape\n    idx = hnswlib.Index(space="ip", dim=d)\n    idx.init_index(max_elements=n, ef_construction=200, M=M, random_seed=seed)\n    idx.add_items(emb, np.arange(n), num_threads=-1)\n    idx.set_ef(max(ef, k + 1))\n    kk = min(k + 1, n)\n    lab, dist = idx.knn_query(emb, k=kk, num_threads=-1)\n    cos = 1.0 - dist\n    src = np.repeat(np.arange(n), kk)\n    dst, c = lab.ravel(), cos.ravel()\n    keep = (src != dst) & (c >= threshold)\n    a, b = np.minimum(src[keep], dst[keep]), np.maximum(src[keep], dst[keep])\n    df = pd.DataFrame({"i": a, "j": b, "cos": c[keep]}).groupby(["i", "j"], as_index=False)["cos"].max()\n    df = df.astype({"i": "int64", "j": "int64", "cos": "float32"})\n    by = {}\n    for t in sorted({threshold, *report_thresholds}):\n        above = (cos[:, 1:] >= t).sum(axis=1) if kk > 1 else np.zeros(n, dtype=int)\n        by[float(t)] = {"n_pairs": int((df["cos"] >= t).sum()), "n_nodes_saturated": int((above >= kk - 1).sum()) if kk > 1 else 0,\n                        "n_nodes_with_neighbor": int((above > 0).sum())}\n    info = {"k": k, "threshold": threshold, "n_pairs": int(len(df)), "n_nodes_saturated": by[float(threshold)]["n_nodes_saturated"],\n            "n_nodes_with_neighbor": by[float(threshold)]["n_nodes_with_neighbor"], "by_threshold": by}\n    return df, info\n\n\ndef exact_neighbors(emb: np.ndarray, query_idx: np.ndarray, threshold: float, exclude_self: bool = True, block: int = 256) -> pd.DataFrame:\n    """Точний пошук: усі вектори з cos >= threshold до кожного запиту (без обмеження на k). Повертає колонки q, i, cos."""\n    rows = []\n    for s in range(0, len(query_idx), block):\n        q = query_idx[s:s + block]\n        sims = emb[q] @ emb.T\n        if exclude_self:\n            sims[np.arange(len(q)), q] = -2.0\n        qi, ii = np.nonzero(sims >= threshold)\n        rows.append(pd.DataFrame({"q": q[qi], "i": ii, "cos": sims[qi, ii]}))\n    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame({"q": [], "i": [], "cos": []})\n\n\ndef hnsw_recall(emb: np.ndarray, found: pd.DataFrame, threshold: float, n_check: int = 500, seed: int = 42, k: int = 80) -> dict:\n    """Повнота hnsw за порогом: для випадкових вузлів порівнюємо число знайдених сусідів з cos >= threshold з точним (обмеженим k)."""\n    rng = np.random.default_rng(seed)\n    q = rng.choice(len(emb), size=min(n_check, len(emb)), replace=False)\n    ex = exact_neighbors(emb, q, threshold)\n    exact_n = ex.groupby("q").size().reindex(q).fillna(0).clip(upper=k).to_numpy()\n    deg = pd.concat([found.groupby("i").size(), found.groupby("j").size()]).groupby(level=0).sum()\n    found_n = deg.reindex(q).fillna(0).clip(upper=k).to_numpy()\n    tot = exact_n.sum()\n    return {"n_queries": int(len(q)), "n_with_neighbors": int((exact_n > 0).sum()), "exact_neighbors_capped": int(tot),\n            "recall": float(np.minimum(found_n, exact_n).sum() / tot) if tot > 0 else float("nan")}\n\n\n# ---------------------------------------------------------------- порівняння з ребрами набору\n\ndef pair_cosines(emb: np.ndarray, key2idx: dict, a_keys: np.ndarray, b_keys: np.ndarray) -> np.ndarray:\n    ia = np.array([key2idx.get(int(x), -1) for x in a_keys])\n    ib = np.array([key2idx.get(int(x), -1) for x in b_keys])\n    ok = (ia >= 0) & (ib >= 0)\n    out = np.full(len(ia), np.nan, dtype=np.float32)\n    out[ok] = np.einsum("ij,ij->i", emb[ia[ok]], emb[ib[ok]])\n    return out\n\n\ndef compare_scores(score: np.ndarray, cos: np.ndarray) -> dict:\n    """Наскільки збережена оцінка схожості набору збігається з нашим косинусом. Моделі могли відрізнятись, тому крім різниці\n    даємо кореляції (інваріантні до монотонного перетворення, зокрема (cos + 1) / 2) і лінійну підгонку score ~ a * cos + b."""\n    m = np.isfinite(score) & np.isfinite(cos)\n    s, c = score[m].astype(float), cos[m].astype(float)\n    a, b = np.polyfit(c, s, 1) if len(s) > 1 else (np.nan, np.nan)\n    d = s - c\n    dm = s - (c + 1) / 2\n    nt = s < 0.995  # майже однакові тексти дають оцінки ~1 з шумом, ранги на них не інформативні: окремі показники для решти пар\n    extra = {"share_score_ge_0995": float((~nt).mean()), "n_nontrivial": int(nt.sum())}\n    if nt.sum() > 2 and np.ptp(s[nt]) > 0 and np.ptp(c[nt]) > 0:\n        extra.update({"pearson_nontrivial": float(pearsonr(s[nt], c[nt])[0]), "spearman_nontrivial": float(spearmanr(s[nt], c[nt])[0]),\n                      "mean_abs_diff_direct_nontrivial": float(np.abs(d[nt]).mean()),\n                      "mean_abs_diff_mapped_half_nontrivial": float(np.abs(dm[nt]).mean())})\n    else:\n        extra.update({k: float("nan") for k in ("pearson_nontrivial", "spearman_nontrivial", "mean_abs_diff_direct_nontrivial", "mean_abs_diff_mapped_half_nontrivial")})\n    return {**extra, "n": int(m.sum()), "pearson": float(pearsonr(s, c)[0]), "spearman": float(spearmanr(s, c)[0]),\n            "mean_abs_diff_direct": float(np.abs(d).mean()), "mean_abs_diff_mapped_half": float(np.abs(dm).mean()),\n            "median_diff_direct": float(np.median(d)), "diff_p05": float(np.quantile(d, 0.05)), "diff_p95": float(np.quantile(d, 0.95)),\n            "fit_slope": float(a), "fit_intercept": float(b), "share_cos_ge_score_min": float((c >= s.min()).mean()),\n            "score_min": float(s.min()), "score_max": float(s.max()), "cos_min": float(c.min()), "cos_max": float(c.max())}\n\n\ndef overlap(own: set, orig: set) -> dict:\n    inter = len(own & orig)\n    return {"n_own": len(own), "n_orig": len(orig), "n_intersection": inter,\n            "recall_of_orig": inter / len(orig) if orig else float("nan"), "share_of_own_in_orig": inter / len(own) if own else float("nan")}\n\n\ndef pair_set(a: np.ndarray, b: np.ndarray) -> set:\n    return {(min(int(x), int(y)), max(int(x), int(y))) for x, y in zip(a, b)}\n\n\n# ---------------------------------------------------------------- копії золотих\n\ndef gold_copies(emb: np.ndarray, emb_set: pd.DataFrame, thresholds=THRESHOLDS) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Для кожного золотого повідомлення всі інші ембедовані повідомлення з cos >= min(thresholds) (точний пошук).\n    Повертає (пари, зведення за порогами). Сусіди, що самі золоті, теж потрапляють (позначено in_gold)."""\n    q = np.flatnonzero(emb_set["in_gold"].to_numpy())\n    pairs = exact_neighbors(emb, q, min(thresholds))\n    key = emb_set["key"].to_numpy()\n    txt = emb_set["text"].to_numpy()\n    pairs = pairs.assign(gold_key=key[pairs["q"]], neighbor_key=key[pairs["i"]],\n                         neighbor_in_labeled=emb_set["in_labeled"].to_numpy()[pairs["i"]],\n                         neighbor_in_gold=emb_set["in_gold"].to_numpy()[pairs["i"]],\n                         same_text=txt[pairs["q"]] == txt[pairs["i"]]).drop(columns=["q", "i"])\n    rows = []\n    for t in thresholds:\n        p = pairs[pairs["cos"] >= t]\n        rows.append({"threshold": t, "n_gold": int(len(q)), "gold_with_neighbor": int(p["gold_key"].nunique()),\n                     "gold_with_labeled_neighbor": int(p.loc[p["neighbor_in_labeled"], "gold_key"].nunique()),\n                     "neighbor_messages": int(p["neighbor_key"].nunique()),\n                     "neighbor_messages_in_labeled": int(p.loc[p["neighbor_in_labeled"], "neighbor_key"].nunique()),\n                     "pairs": int(len(p)), "pairs_exact_text": int(p["same_text"].sum())})\n    return pairs, pd.DataFrame(rows)\n\n\n# ---------------------------------------------------------------- досяжність\n\ndef _adj(n: int, a: np.ndarray, b: np.ndarray) -> sparse.csr_matrix:\n    m = sparse.coo_matrix((np.ones(len(a), dtype=np.float32), (a, b)), shape=(n, n)).tocsr()\n    m = ((m + m.T) > 0).astype(np.float32)\n    m.setdiag(0)\n    m.eliminate_zeros()\n    return m.tocsr()\n\n\ndef reach_within(adj: sparse.csr_matrix, source_mask: np.ndarray, kmax: int) -> list[np.ndarray]:\n    """x_k[v] = True, якщо від v можна дійти до вузла з source_mask не більш ніж за k кроків (неспрямований граф)."""\n    x = source_mask.astype(np.float32)\n    out = []\n    for _ in range(kmax):\n        x = ((x + adj @ x) > 0).astype(np.float32)\n        out.append(x > 0)\n    return out\n\n\ndef reachability(n_nodes: int, a: np.ndarray, b: np.ndarray, group: np.ndarray, kmax: int = 4) -> pd.DataFrame:\n    """Для вузлів групи h частка тих, від кого за <= k кроків можна дійти до вузла іншої групи.\n    group: bool на вузол (True = канал групи n), вузли поза шаром (ступінь 0) не враховуються."""\n    adj = _adj(n_nodes, a, b)\n    present = np.asarray(adj.sum(axis=1)).ravel() > 0\n    rows = []\n    for name, h in (("n_to_nn", True), ("nn_to_n", False)):\n        start = present & (group == h)\n        reach = reach_within(adj, present & (group != h), kmax)\n        for k, r in enumerate(reach, 1):\n            rows.append({"direction": name, "k": k, "n_start": int(start.sum()), "n_reach": int((r & start).sum()),\n                         "share": float((r & start).sum() / start.sum()) if start.sum() else float("nan")})\n    return pd.DataFrame(rows)\n\n\ndef reachability_null(n_nodes: int, a: np.ndarray, b: np.ndarray, channel_of: np.ndarray, channel_group: dict, kmax: int = 4,\n                      n_perm: int = 20, seed: int = 42) -> pd.DataFrame:\n    """Нульова модель: групи розкидані по каналах випадково (розміри груп збережено), повідомлення успадковують групу каналу."""\n    rng = np.random.default_rng(seed)\n    chs = np.array(sorted(channel_group))\n    vals = np.array([channel_group[c] for c in chs])\n    pos = {c: i for i, c in enumerate(chs)}\n    ci = np.array([pos.get(int(c), -1) for c in channel_of])\n    ok = ci >= 0\n    acc = []\n    for _ in range(n_perm):\n        g = np.zeros(n_nodes, dtype=bool)\n        g[ok] = rng.permutation(vals)[ci[ok]]\n        acc.append(reachability(n_nodes, a, b, g, kmax).set_index(["direction", "k"])["share"])\n    df = pd.concat(acc, axis=1)\n    return pd.DataFrame({"null_mean": df.mean(axis=1), "null_p05": df.quantile(0.05, axis=1), "null_p95": df.quantile(0.95, axis=1)}).reset_index()\n\n\n# ---------------------------------------------------------------- перевірка форвардів проти channel_flow (залишок S3)\n\ndef forward_vs_channel_flow(fwd: pd.DataFrame, flow: pd.DataFrame) -> dict:\n    """Агрегує message-рівневі forward-ребра до пар каналів і звіряє з таблицею channel_flow_edges (S3)."""\n    f = fwd.rename(columns={"logical_src_channel_id": "s", "logical_dst_channel_id": "d"})[["s", "d"]]\n    n_all = len(f)\n    loops = f["s"] == f["d"]\n    agg = f[~loops].groupby(["s", "d"]).size().rename("w").reset_index()\n    fl = flow.rename(columns={"src_channel_id": "s", "dst_channel_id": "d", "weight": "wf"})[["s", "d", "wf"]]\n    mm = agg.merge(fl, on=["s", "d"], how="outer").fillna(0)\n    diff = mm[mm["w"] != mm["wf"]]\n    return {"n_forward_edges": int(n_all), "n_self_forward": int(loops.sum()), "n_channel_pairs_from_forward": int(len(agg)),\n            "weight_sum_without_self": int(agg["w"].sum()), "channel_flow_rows": int(len(fl)), "channel_flow_weight_sum": int(fl["wf"].sum()),\n            "n_pairs_with_different_weight": int(len(diff)), "equal_to_channel_flow": bool(len(diff) == 0)}\n\n\ndef pair_group_stats(ch_a: np.ndarray, ch_b: np.ndarray, group: dict) -> dict:\n    """Частки пар за групами каналів (n, nn) і частка пар усередині одного каналу. Пари з каналом без групи рахуються окремо."""\n    ga = pd.Series(ch_a).map(group)\n    gb = pd.Series(ch_b).map(group)\n    known = ga.notna().to_numpy() & gb.notna().to_numpy()\n    a, b = ga[known].astype(bool).to_numpy(), gb[known].astype(bool).to_numpy()\n    n = int(known.sum())\n    out = {"n_pairs": int(len(ch_a)), "n_unknown_group": int((~known).sum()), "same_channel_share": float((np.asarray(ch_a) == np.asarray(ch_b)).mean()) if len(ch_a) else float("nan")}\n    out.update({"n_n": int((a & b).sum()), "n_nn": int((a != b).sum()), "nn_nn": int((~a & ~b).sum()),\n                "cross_share": float((a != b).sum() / n) if n else float("nan")})\n    return out\n', 'src/narr/semantic_viz.py': '"""Графіки S4: відтворення оцінок схожості, досяжність між групами, UMAP. Підписи українською, 150 dpi."""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt  # noqa: E402\nimport numpy as np  # noqa: E402\nimport pandas as pd  # noqa: E402\n\nfrom narr.viz import C_ACC, C_NEG, C_POS, _style  # noqa: E402\n\nLAYER_COLORS = {"forward": "#2c3e50", "semantic_orig": "#c0392b", "semantic_own_0985": "#e67e22", "semantic_own_0975": "#f1c40f",\n                "forward+semantic_orig": "#16a085"}\nLAYER_LABELS = {"forward": "форварди", "semantic_orig": "семантичні ребра набору (від 0,975)",\n                "semantic_own_0985": "наші семантичні ребра (від 0,985)", "semantic_own_0975": "наші семантичні ребра (від 0,975)",\n                "forward+semantic_orig": "форварди і семантичні ребра набору"}\n\n\ndef fig_repro(sample: pd.DataFrame, stats: dict, out: Path) -> None:\n    """sample: колонки score (з набору) і cos (наш bge-m3)."""\n    _style()\n    fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))\n    ax[0].scatter(sample["cos"], sample["score"], s=4, alpha=0.25, color=C_ACC)\n    lo, hi = float(min(sample["cos"].min(), sample["score"].min())), 1.0\n    xs = np.linspace(lo, hi, 50)\n    ax[0].plot(xs, xs, color=C_POS, lw=1.2, label="score = cos")\n    ax[0].plot(xs, (xs + 1) / 2, color=C_NEG, lw=1.2, ls="--", label="score = (cos + 1) / 2")\n    ax[0].set(xlabel="наш косинус (bge-m3)", ylabel="similarity_score з набору", title="Пари семантичних ребер набору")\n    ax[0].legend(frameon=False)\n    d = (sample["score"] - sample["cos"]).to_numpy()\n    ax[1].hist(d, bins=60, color="#8aa9d6")\n    ax[1].axvline(0, color=C_POS, lw=1)\n    ax[1].set(xlabel="score - cos", ylabel="пар", title=f"Різниця (спірмен {stats[\'spearman\']:.3f}, середній модуль {stats[\'mean_abs_diff_direct\']:.3f})")\n    fig.tight_layout()\n    fig.savefig(out)\n    plt.close(fig)\n\n\ndef fig_reach(reach: pd.DataFrame, out: Path) -> None:\n    """reach: layer, direction (n_to_nn, nn_to_n, all), k, share, null_mean (може бути NaN)."""\n    _style()\n    dirs = [("n_to_nn", "від каналів n до груп nn"), ("nn_to_n", "від каналів nn до груп n"), ("all", "усі повідомлення шару")]\n    fig, axs = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)\n    for ax, (d, title) in zip(axs, dirs):\n        for layer, g in reach[reach["direction"] == d].groupby("layer"):\n            g = g.sort_values("k")\n            col = LAYER_COLORS.get(layer, "#7f8c8d")\n            ax.plot(g["k"], g["share"], marker="o", color=col, label=LAYER_LABELS.get(layer, layer))\n            if g["null_mean"].notna().any():\n                ax.plot(g["k"], g["null_mean"], ls=":", color=col, alpha=0.7)\n        ax.set(xlabel="кроків", title=title, xticks=sorted(reach["k"].unique()), ylim=(0, 1))\n    axs[0].set_ylabel("частка повідомлень, що досягають іншої групи")\n    axs[0].legend(frameon=False, fontsize=7, loc="lower right")\n    fig.text(0.5, 0.005, "Пунктир: нульова модель (групи випадково розкидані по каналах)", ha="center", fontsize=8)\n    fig.tight_layout(rect=(0, 0.07, 1, 1))\n    fig.savefig(out)\n    plt.close(fig)\n\n\ndef fig_umap(c: pd.DataFrame, out: Path, top_meta: int = 6) -> None:\n    """c: x, y, source (gold / graph), language, meta (текст або \'без наративу\')."""\n    _style()\n    fig, axs = plt.subplots(1, 3, figsize=(16, 5.2))\n    metas = c.loc[c["meta"] != "без наративу", "meta"].value_counts()\n    keep = list(metas.index[:top_meta])\n    cmap = plt.get_cmap("tab10")\n    mcol = {m: cmap(i) for i, m in enumerate(keep)}\n    none = c["meta"] == "без наративу"\n    axs[0].scatter(c.loc[none, "x"], c.loc[none, "y"], s=2, color="#d5d8dc", alpha=0.5, label="без наративу")\n    for m in keep:\n        s = c["meta"] == m\n        axs[0].scatter(c.loc[s, "x"], c.loc[s, "y"], s=3, color=mcol[m], alpha=0.8, label=f"{m[:42]} ({int(s.sum())})")\n    oth = ~none & ~c["meta"].isin(keep)\n    axs[0].scatter(c.loc[oth, "x"], c.loc[oth, "y"], s=3, color="#7f8c8d", alpha=0.8, label=f"інші наративи ({int(oth.sum())})")\n    axs[0].set_title("Мета-наратив (слабка мітка або золота)")\n    axs[0].legend(frameon=False, fontsize=6, markerscale=3, loc="upper left")\n    lcol = {"uk": "#2980b9", "ru": "#c0392b", "en": "#27ae60"}\n    for lg, col in lcol.items():\n        s = c["language"] == lg\n        axs[1].scatter(c.loc[s, "x"], c.loc[s, "y"], s=2, color=col, alpha=0.6, label=f"{lg} ({int(s.sum())})")\n    axs[1].set_title("Мова (langid)")\n    axs[1].legend(frameon=False, fontsize=7, markerscale=4)\n    g = c["source"] == "gold"\n    axs[2].scatter(c.loc[~g, "x"], c.loc[~g, "y"], s=2, color="#d5d8dc", alpha=0.6, label=f"вибірка графа ({int((~g).sum())})")\n    pos = g & (c["meta"] != "без наративу")\n    axs[2].scatter(c.loc[g & ~pos, "x"], c.loc[g & ~pos, "y"], s=10, color=C_NEG, edgecolor="white", lw=0.3, label=f"золоті без наративу ({int((g & ~pos).sum())})")\n    axs[2].scatter(c.loc[pos, "x"], c.loc[pos, "y"], s=10, color=C_POS, edgecolor="white", lw=0.3, label=f"золоті наративні ({int(pos.sum())})")\n    axs[2].set_title("Золоті повідомлення на тлі вибірки графа")\n    axs[2].legend(frameon=False, fontsize=7, markerscale=2)\n    for ax in axs:\n        ax.set_xticks([])\n        ax.set_yticks([])\n        for sp in ax.spines.values():\n            sp.set_visible(False)\n    fig.tight_layout()\n    fig.savefig(out)\n    plt.close(fig)\n', 'configs/base.yaml': '# Базова конфігурація проєкту. Усі шляхи відносно кореня репозиторію.\nseed: 42\n\npaths:\n  telenarratives: external/TeleNarratives\n  golden: "external/TeleNarratives/labeling results/strong labeling/golden set (labeled).csv"\n  shared: "external/TeleNarratives/labeling results/strong labeling/shared subset (labeled).csv"\n  iaa: "external/TeleNarratives/labeling results/strong labeling/inter_annotator_agreement_metrics.csv"\n  channels: external/TeleNarratives/config/channels.csv\n  narratives: external/TeleNarratives/config/narratives.csv\n  important_dates: "external/TeleNarratives/config/Important dates.csv"\n  gnn_results: "external/TeleNarratives/gnn results"\n  weak_eval: "external/TeleNarratives/labeling results/weak labeling/evaluation"\n\ncolumns:\n  no_narrative_label: "Не містить наративу"\n  binary: "Binary label"\n  general: "General label"\n  level1: "General label (level-1)"\n  text: text\n  channel: channel_id\n  message: message_id\n  date: date_utc\n  forwarded: is_forwarded\n  reviewers: ["Reviewer 1", "Reviewer 2", "Reviewer 3"]\n  # у файлі третій рецензент на рівні level-1 названий "Reviewer 2 (level-1).1"\n  reviewers_level1: ["Reviewer 1 (level-1)", "Reviewer 2 (level-1)", "Reviewer 2 (level-1).1"]\n\n# Очікувані значення з PLAN_vykonannya.md, розділ 2.2. Перевіряються тестами і аудитом.\nexpected:\n  n_messages: 412\n  n_narrative: 148\n  n_shared: 103\n  n_channels_config: 98\n  n_sub_narratives: 380\n  n_channels_in_gold: 56\n\neval:\n  bootstrap_n: 1000\n  cv_folds: 5\n  cv_repeats: 5\n  n_top_meta_narratives: 4   # багатокласова оцінка: none + 4 найбільші + «інше»\n  threshold: 0.5\n', 'scripts/s04_semantic.py': '"""S4: семантичний шар. Ембединги bge-m3, власні семантичні ребра, порівняння з ребрами набору, копії золотих, досяжність, UMAP.\n\nЗапуск (Colab з GPU):\n  python3 -I scripts/s04_semantic.py --raw-dir <data/raw> --out <results/04_semantic> --interim <data/interim/embeddings>\nБез GPU і без bge-m3 можна перевірити логіку: `--encoder hash --skip-umap` (хеш-ембединги, лише для тестів).\n\nЩо робить: (1) збирає набір повідомлень для ембедингів (золоті, розмічена партія, кінці семантичних ребер набору), (2) кодує їх bge-m3\nу float16 і кешує на диску, (3) звіряє наш косинус з `similarity_score` ребер набору, (4) будує власні ребра hnswlib (k=80),\n(5) шукає точні копії золотих, (6) рахує досяжність між групами каналів для форвардів і семантики, (7) UMAP,\n(8) звіряє forward_edges з channel_flow_edges (залишок S3). Код завершується з ненульовим статусом, якщо порушено жорсткий інваріант.\n"""\nimport argparse\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))\n\nfrom narr import eda, semantic as sm  # noqa: E402\nfrom narr.eda import KEY_MULT  # noqa: E402\nfrom narr.io import ROOT, load_config, path_of, write_run_json  # noqa: E402\n\nKMAX = 4\n\n\ndef th(n) -> str:\n    return f"{int(n):,}".replace(",", " ")\n\nLAYER_ORDER = ["forward", "semantic_orig", "semantic_own_0985", "semantic_own_0975", "forward+semantic_orig"]\n\n\ndef make_bge_encoder(max_len: int, device: str):\n    from sentence_transformers import SentenceTransformer\n\n    model = SentenceTransformer(sm.MODEL_NAME, device=device)\n    model.max_seq_length = max_len\n    if device == "cuda":\n        model.half()\n\n    def enc(batch):\n        return model.encode(batch, batch_size=len(batch), normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)\n\n    return enc, model.tokenizer\n\n\ndef token_stats(tokenizer, texts: list[str], max_len: int, chunk: int = 10_000) -> dict:\n    lens = []\n    for s in range(0, len(texts), chunk):\n        lens.extend(len(x) for x in tokenizer(texts[s:s + chunk], add_special_tokens=True, truncation=False)["input_ids"])\n    a = np.array(lens)\n    return {"tokens_median": float(np.median(a)), "tokens_p95": float(np.quantile(a, 0.95)), "tokens_max": int(a.max()),\n            "max_len": max_len, "share_truncated": float((a > max_len).mean())}\n\n\ndef get_embeddings(es: pd.DataFrame, args) -> tuple[np.ndarray, dict]:\n    keys = es["key"].to_numpy()\n    interim = Path(args.interim) if args.interim else None\n    if interim is not None and (interim / "bge_m3_keys.npy").exists() and args.encoder == "bge-m3":\n        k0 = np.load(interim / "bge_m3_keys.npy")\n        if len(k0) == len(keys) and (k0 == keys).all():\n            print(f"Ембединги з кешу: {interim}", flush=True)\n            return sm.quantize_fp16(np.load(interim / "bge_m3_fp16.npy").astype(np.float32)), {"from_cache": True, "encoder": args.encoder}\n        print("Кеш не збігається з поточним набором ключів, рахую заново", flush=True)\n    texts = es["text"].tolist()\n    info = {"from_cache": False, "encoder": args.encoder}\n    t0 = time.time()\n    if args.encoder == "bge-m3":\n        import torch\n\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n        if device == "cpu" and not args.allow_cpu:\n            raise SystemExit("GPU не знайдено. У Colab: Runtime -> Change runtime type -> T4 GPU (або запустіть з --allow-cpu, але це годинами).")\n        enc, tok = make_bge_encoder(args.max_len, device)\n        info.update({"model": sm.MODEL_NAME, "device": device, "batch_size": args.batch_size, **token_stats(tok, texts, args.max_len)})\n    else:\n        enc = sm.hash_encoder()\n        info.update({"model": "hash-trigrams (лише тест)", "device": "cpu", "batch_size": args.batch_size})\n    emb = sm.embed_sorted(texts, enc, args.batch_size,\n                          progress=lambda d, n: print(f"  закодовано {th(d)} з {th(n)}, {time.time() - t0:.0f} с", flush=True))\n    emb = sm.quantize_fp16(emb)\n    info["seconds"] = round(time.time() - t0)\n    if interim is not None and args.encoder == "bge-m3":\n        interim.mkdir(parents=True, exist_ok=True)\n        np.save(interim / "bge_m3_fp16.npy", emb.astype(np.float16))\n        np.save(interim / "bge_m3_keys.npy", keys)\n        es.drop(columns=["text"]).to_parquet(interim / "bge_m3_index.parquet")\n        print(f"Збережено ембединги: {interim}", flush=True)\n    return emb, info\n\n\ndef main(argv=None) -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--raw-dir", required=True)\n    ap.add_argument("--out", default=None)\n    ap.add_argument("--interim", default=None, help="папка для кешу ембедингів (float16 .npy), не потрапляє в репозиторій")\n    ap.add_argument("--golden", default=None)\n    ap.add_argument("--per-channel", default=str(ROOT / "results" / "02_eda" / "per_channel.csv"))\n    ap.add_argument("--channel-flow", default=str(ROOT / "data_small" / "channel_flow_edges.csv"))\n    ap.add_argument("--encoder", choices=["bge-m3", "hash"], default="bge-m3")\n    ap.add_argument("--allow-cpu", action="store_true")\n    ap.add_argument("--batch-size", type=int, default=128)\n    ap.add_argument("--max-len", type=int, default=512)\n    ap.add_argument("--k", type=int, default=80)\n    ap.add_argument("--umap-n", type=int, default=20_000)\n    ap.add_argument("--skip-umap", action="store_true")\n    ap.add_argument("--n-perm", type=int, default=20)\n    args = ap.parse_args(argv)\n\n    cfg = load_config()\n    seed = cfg["seed"]\n    out = Path(args.out) if args.out else ROOT / "results" / "04_semantic"\n    out.mkdir(parents=True, exist_ok=True)\n    golden = Path(args.golden) if args.golden else path_of(cfg, "golden")\n    t0 = time.time()\n\n    print("Читаю дані ...", flush=True)\n    inp = sm.load_inputs(args.raw_dir, golden)\n    es, set_info = sm.build_embedding_set(inp["msgs"], inp["tg"], inp["gold"], inp["sem"], inp["fwd"])\n    del inp["msgs"], inp["tg"]  # великі тексти далі не потрібні (економія пам\'яті)\n    key = es["key"].to_numpy()\n    key2idx = {int(k): i for i, k in enumerate(key)}\n    print(f"  до ембедингів {th(len(es))} повідомлень (золотих {set_info[\'n_gold\']}, розмічених {th(set_info[\'n_labeled\'])}, "\n          f"кінців семантичних ребер {th(set_info[\'n_sem_endpoints\'])}), порожніх відкинуто {set_info[\'n_empty_dropped\']}", flush=True)\n\n    emb, enc_info = get_embeddings(es, args)\n    norms = np.linalg.norm(emb, axis=1)\n    enc_info.update({"n": int(len(emb)), "dim": int(emb.shape[1]), "max_abs_norm_dev": float(np.abs(norms - 1).max()),\n                     "all_finite": bool(np.isfinite(emb).all())})\n\n    pc = pd.read_csv(args.per_channel)\n    channel_group = {int(c): bool(v) for c, v in zip(pc["channel_id"], pc["table_narrative_presence"])}\n    sem = inp["sem"]\n    ka = eda.keys(sem, "src_channel_id", "src_message_id").to_numpy()\n    kb = eda.keys(sem, "dst_channel_id", "dst_message_id").to_numpy()\n    score = sem["similarity_score"].to_numpy()\n\n    # ---- (3) відтворення оцінок набору\n    cos_orig = sm.pair_cosines(emb, key2idx, ka, kb)\n    repro = sm.compare_scores(score, cos_orig)\n    repro.update({"n_pairs_total": int(len(sem)), "n_pairs_without_cos": int(np.isnan(cos_orig).sum())})\n    rng = np.random.default_rng(seed)\n    samp = rng.choice(len(sem), size=min(8000, len(sem)), replace=False)\n    sample = pd.DataFrame({"score": score[samp], "cos": cos_orig[samp]}).dropna()\n\n    # ---- (4) власні ребра\n    print("Будую hnsw-ребра ...", flush=True)\n    pairs, kinfo = sm.knn_pairs(emb, k=args.k, threshold=min(sm.THRESHOLDS), report_thresholds=sm.THRESHOLDS, seed=seed)\n    pairs = pairs.assign(key_a=key[pairs["i"]], key_b=key[pairs["j"]])\n    own = {t: pairs[pairs["cos"] >= t] for t in sm.THRESHOLDS}\n    recall = {str(t): sm.hnsw_recall(emb, own[t], t, n_check=500, seed=seed, k=args.k) for t in (0.975, 0.985)}\n    orig_set = sm.pair_set(ka, kb)\n    sem_node = es["in_sem"].to_numpy()\n    ov, ov_sem = {}, {}\n    for t in sm.THRESHOLDS:\n        ps = sm.pair_set(own[t]["key_a"].to_numpy(), own[t]["key_b"].to_numpy())\n        ov[str(t)] = sm.overlap(ps, orig_set)\n        both = own[t][sem_node[own[t]["i"].to_numpy()] & sem_node[own[t]["j"].to_numpy()]]\n        ov_sem[str(t)] = sm.overlap(sm.pair_set(both["key_a"].to_numpy(), both["key_b"].to_numpy()), orig_set)\n    pg = {"orig": sm.pair_group_stats(ka // KEY_MULT, kb // KEY_MULT, channel_group)}\n    for t in (0.975, 0.985):\n        pg[f"own_{t}"] = sm.pair_group_stats(own[t]["key_a"].to_numpy() // KEY_MULT, own[t]["key_b"].to_numpy() // KEY_MULT, channel_group)\n\n    # ---- (5) копії золотих\n    print("Шукаю копії золотих ...", flush=True)\n    gpairs, gsumm = sm.gold_copies(emb, es)\n    excl = (gpairs[(gpairs["cos"] >= 0.985) & gpairs["neighbor_in_labeled"] & ~gpairs["neighbor_in_gold"]]\n            .groupby("neighbor_key").agg(max_cos=("cos", "max"), n_gold_matched=("gold_key", "nunique")).reset_index()\n            .rename(columns={"neighbor_key": "key"}))\n    excl["channel_id"], excl["message_id"] = excl["key"] // KEY_MULT, excl["key"] % KEY_MULT\n    same_text_min = float(gpairs.loc[gpairs["same_text"], "cos"].min()) if gpairs["same_text"].any() else float("nan")\n\n    # ---- (6) досяжність\n    print("Рахую досяжність ...", flush=True)\n    fa = eda.keys(inp["fwd"], "logical_src_channel_id", "logical_src_message_id").to_numpy()\n    fb = eda.keys(inp["fwd"], "logical_dst_channel_id", "logical_dst_message_id").to_numpy()\n    layers = {"forward": (fa, fb), "semantic_orig": (ka, kb),\n              "semantic_own_0985": (own[0.985]["key_a"].to_numpy(), own[0.985]["key_b"].to_numpy()),\n              "semantic_own_0975": (own[0.975]["key_a"].to_numpy(), own[0.975]["key_b"].to_numpy()),\n              "forward+semantic_orig": (np.concatenate([fa, ka]), np.concatenate([fb, kb]))}\n    universe = np.unique(np.concatenate([np.concatenate(v) for v in layers.values()]))\n    ch_of = universe // KEY_MULT\n    gser = pd.Series(ch_of).map(channel_group)\n    known = gser.notna().to_numpy()\n    garr = gser.fillna(False).astype(bool).to_numpy()\n    reach_rows, dropped = [], {}\n    for name in LAYER_ORDER:\n        a, b = (np.searchsorted(universe, x) for x in layers[name])\n        ok = known[a] & known[b]\n        dropped[name] = int((~ok).sum())\n        a, b = a[ok], b[ok]\n        r = sm.reachability(len(universe), a, b, garr, KMAX)\n        nl = sm.reachability_null(len(universe), a, b, ch_of, channel_group, KMAX, n_perm=args.n_perm, seed=seed)\n        r = r.merge(nl, on=["direction", "k"], how="left")\n        tot = r.groupby("k").agg(n_start=("n_start", "sum"), n_reach=("n_reach", "sum")).reset_index()\n        tot["direction"], tot["share"] = "all", tot["n_reach"] / tot["n_start"]\n        r = pd.concat([r, tot], ignore_index=True)\n        r["layer"] = name\n        r["n_edges"] = int(len(a))\n        reach_rows.append(r)\n    reach = pd.concat(reach_rows, ignore_index=True)[["layer", "direction", "k", "n_start", "n_reach", "share", "null_mean", "null_p05", "null_p95", "n_edges"]]\n\n    # ---- 50 пар біля порогу\n    near = sem[(sem["similarity_score"] >= 0.975) & (sem["similarity_score"] < 0.985)].copy()\n    near["cos_ours"] = sm.pair_cosines(emb, key2idx, eda.keys(near, "src_channel_id", "src_message_id").to_numpy(),\n                                       eda.keys(near, "dst_channel_id", "dst_message_id").to_numpy())\n    near = near.sample(n=min(50, len(near)), random_state=seed).reset_index(drop=True)\n    txt = dict(zip(es["key"], es["text"]))\n    near["text_src"] = [txt.get(int(k), "")[:300] for k in eda.keys(near, "src_channel_id", "src_message_id")]\n    near["text_dst"] = [txt.get(int(k), "")[:300] for k in eda.keys(near, "dst_channel_id", "dst_message_id")]\n    near["same_channel"] = near["src_channel_id"] == near["dst_channel_id"]\n\n    # ---- (8) залишок S3\n    flow = pd.read_csv(args.channel_flow)\n    fwd_check = sm.forward_vs_channel_flow(inp["fwd"], flow)\n\n    # ---- перевірки\n    gold_ok = set_info["n_gold"] == len(inp["gold"].drop_duplicates(["channel_id", "message_id"])) and set_info["n_gold_empty"] == 0\n    miss_share = repro["n_pairs_without_cos"] / max(repro["n_pairs_total"], 1)\n    checks = [\n        {"hard": True, "check": "Усі золоті повідомлення ембедовані, порожніх текстів немає", "passed": bool(gold_ok),\n         "detail": f"{set_info[\'n_gold\']} золотих, порожніх {set_info[\'n_gold_empty\']}"},\n        {"hard": True, "check": "Ключі набору унікальні", "passed": bool(len(key) == len(set(key.tolist()))), "detail": f"{th(len(key))} ключів"},\n        {"hard": True, "check": "Ембединги скінченні й одиничної довжини (відхилення норми менше 1e-3)", "passed": bool(enc_info["all_finite"] and enc_info["max_abs_norm_dev"] < 1e-3),\n         "detail": f"макс. відхилення {enc_info[\'max_abs_norm_dev\']:.2e}, розмірність {enc_info[\'dim\']}"},\n        {"hard": True, "check": "Для 99% і більше семантичних ребер набору є наш косинус (обидва кінці ембедовані)", "passed": bool(miss_share <= 0.01),\n         "detail": f"без косинуса {repro[\'n_pairs_without_cos\']} з {th(repro[\'n_pairs_total\'])}"},\n        {"hard": True, "check": "Повнота hnsw за порогом 0,985 не нижча за 0,98 (500 випадкових вузлів, точний пошук)", "passed": bool(recall["0.985"]["recall"] >= 0.98 or np.isnan(recall["0.985"]["recall"])),\n         "detail": f"{recall[\'0.985\'][\'recall\']:.4f} ({recall[\'0.985\'][\'exact_neighbors_capped\']} сусідів)"},\n        {"hard": True, "check": "Повнота hnsw за порогом 0,975 не нижча за 0,98", "passed": bool(recall["0.975"]["recall"] >= 0.98 or np.isnan(recall["0.975"]["recall"])),\n         "detail": f"{recall[\'0.975\'][\'recall\']:.4f} ({recall[\'0.975\'][\'exact_neighbors_capped\']} сусідів)"},\n        {"hard": False, "check": "Однакові тексти мають косинус не нижчий за 0,995", "passed": bool(np.isnan(same_text_min) or same_text_min >= 0.995),\n         "detail": f"мінімум {same_text_min:.4f}" if not np.isnan(same_text_min) else "пар з однаковим текстом серед копій золотих немає"},\n        {"hard": False, "check": "Оцінки набору збігаються з нашим косинусом за рангами на нетривіальних парах (score < 0,995; спірмен не нижчий за 0,9)",\n         "passed": bool(repro["spearman_nontrivial"] >= 0.9),\n         "detail": f"спірмен {repro[\'spearman_nontrivial\']:.3f}, пірсон {repro[\'pearson_nontrivial\']:.3f}, пар {th(repro[\'n_nontrivial\'])}, пар з score від 0,995 {repro[\'share_score_ge_0995\'] * 100:.1f}%"},\n        {"hard": False, "check": "Оцінки набору збігаються з нашим косинусом за значенням (середній модуль різниці менший за 0,01)", "passed": bool(repro["mean_abs_diff_direct"] < 0.01),\n         "detail": f"усі пари {repro[\'mean_abs_diff_direct\']:.4f}, нетривіальні {repro[\'mean_abs_diff_direct_nontrivial\']:.4f} (з відображенням (cos + 1) / 2: {repro[\'mean_abs_diff_mapped_half\']:.4f})"},\n        {"hard": False, "check": "Агреговані forward_edges збігаються з channel_flow_edges", "passed": bool(fwd_check["equal_to_channel_flow"]),\n         "detail": f"forward {th(fwd_check[\'n_forward_edges\'])}, самопересилань {fwd_check[\'n_self_forward\']}, сума без них {th(fwd_check[\'weight_sum_without_self\'])}, у таблиці {th(fwd_check[\'channel_flow_weight_sum\'])}"},\n    ]\n\n    summary = {"embedding_set": set_info, "encoder": enc_info, "reproduction": repro, "own_edges": {**kinfo, "hnsw_recall": recall},\n               "overlap_with_orig_all_embedded": ov, "overlap_with_orig_within_sem_nodes": ov_sem, "pair_groups": pg,\n               "gold_copies_same_text_min_cos": same_text_min, "n_excluded_labeled_cos985": int(len(excl)),\n               "layers_dropped_edges_unknown_group": dropped, "forward_vs_channel_flow": fwd_check, "n_orig_pairs_near_threshold": int(len(near)),\n               "seconds": round(time.time() - t0)}\n\n    tables = {"reachability": reach, "gold_copies_summary": gsumm, "gold_copies_pairs": gpairs[gpairs["cos"] >= 0.95].sort_values(["gold_key", "cos"], ascending=[True, False]),\n              "exclude_labeled_keys_cos985": excl.sort_values("key"), "threshold_pairs_sample": near, "repro_sample": sample}\n\n    # ---- (7) UMAP\n    if not args.skip_umap:\n        print("UMAP ...", flush=True)\n        try:\n            import umap\n\n            from narr.language import detect_language_parallel\n            pool = np.flatnonzero((es["in_fwd"] | es["in_sem"]).to_numpy() & ~es["in_gold"].to_numpy())\n            pick = np.sort(np.random.default_rng(seed).choice(pool, size=min(args.umap_n, len(pool)), replace=False))\n            gi = np.flatnonzero(es["in_gold"].to_numpy())\n            sel = np.concatenate([gi, pick])\n            xy = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=seed).fit_transform(emb[sel])\n            lang = detect_language_parallel(es["text"].to_numpy()[sel], n_jobs=2)\n            lvl = inp["coding"].drop_duplicates("narrative_id").set_index("narrative_id")["level_1"]\n            gold_meta = inp["gold"].assign(key=eda.keys(inp["gold"])).drop_duplicates("key").set_index("key")["General label (level-1)"]\n            nar = es["narrative_id"].to_numpy()[sel]\n            meta = np.array([lvl.get(int(v), "без наративу") if pd.notna(v) else "без наративу" for v in nar], dtype=object)\n            src = np.where(es["in_gold"].to_numpy()[sel], "gold", "graph")\n            gm = es["key"].to_numpy()[sel][src == "gold"]\n            meta[src == "gold"] = [gold_meta.get(int(k), "без наративу") for k in gm]\n            meta = np.where(meta == cfg["columns"]["no_narrative_label"], "без наративу", meta)\n            coords = pd.DataFrame({"key": es["key"].to_numpy()[sel], "x": xy[:, 0], "y": xy[:, 1], "source": src, "language": lang, "meta": meta})\n            tables["umap_coords"] = coords\n            from narr.semantic_viz import fig_umap\n            (out / "figures").mkdir(exist_ok=True)\n            fig_umap(coords, out / "figures" / "03_umap.png")\n            summary["umap"] = {"n_points": int(len(coords)), "n_gold": int((src == "gold").sum()), "n_graph_sample": int((src == "graph").sum())}\n        except ImportError as e:\n            print(f"UMAP пропущено: {e}", flush=True)\n            summary["umap"] = {"skipped": str(e)}\n\n    eda.save_all(tables, summary, checks, out)\n    from narr.semantic_viz import fig_reach, fig_repro\n    (out / "figures").mkdir(exist_ok=True)\n    fig_repro(sample, repro, out / "figures" / "01_repro_scores.png")\n    fig_reach(reach, out / "figures" / "02_reachability.png")\n    from importlib import metadata\n\n    libs = {}\n    for lib in ("torch", "sentence-transformers", "transformers", "hnswlib", "umap-learn", "scipy", "langid"):\n        try:\n            libs[lib] = metadata.version(lib)\n        except metadata.PackageNotFoundError:\n            libs[lib] = None\n    write_run_json(out, "scripts/s04_semantic.py", {"libs": libs, "raw_dir": str(args.raw_dir), "encoder": args.encoder, "k": args.k, "max_len": args.max_len,\n                                                    "batch_size": args.batch_size, "seed": seed, "umap_n": args.umap_n, "n_perm": args.n_perm,\n                                                    "model": sm.MODEL_NAME, "seconds": round(time.time() - t0)})\n    bad = [c for c in checks if c["hard"] and not c["passed"]]\n    for c in checks:\n        mark = "ok  " if c["passed"] else ("ПОМИЛКА" if c["hard"] else "розбіжність")\n        print(f"[{mark}] {c[\'check\']}: {c[\'detail\']}")\n    if bad:\n        print(f"\\nЖорстких порушень: {len(bad)}")\n        return 1\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', 'results/02_eda/per_channel.csv': 'channel_id,username,table_narrative_presence,table_status,table_label,table_labeled,table_narrative,table_share,n_messages,n_labeled,n_narrative,median_length,forwarded_share,reply_share,share_recomputed,counts_match\n1175084215,ukr_2025_ru,True,labeled,n,892,744,0.8340807174887892,56908,892,744,238.0,0.010947494201166795,0.2763056160820974,0.8340807174887892,True\n1101170442,rian_ru,True,labeled,n,1232,1005,0.8157467532467533,54130,1232,1005,211.0,0.0,0.13212636246074266,0.8157467532467533,True\n1194762769,RBC_ua_news,False,labeled,nn,763,163,0.2136304062909567,45876,763,163,302.0,0.005754266657948081,0.16604546742518364,0.21363040629095675,True\n1283178270,uaobozrevatel,False,labeled,nn,125,6,0.048,45033,125,6,400.0,0.0005329306746014123,0.07962872496336101,0.048,True\n1236359991,novynylive,False,labeled,nn,150,13,0.0866666666666666,41357,150,13,478.0,0.020794545058877578,0.2564015765166719,0.08666666666666667,True\n1081358729,nvua_official,False,labeled,nn,114,8,0.0701754385964912,38817,114,8,366.0,0.004121799165335669,0.2273429852130455,0.07017543859649122,True\n1197363285,u_now,False,labeled,nn,30783,8531,0.277133482766462,31676,30783,8531,321.0,0.0012312160626341709,0.37886728122237656,0.277133482766462,True\n1223955273,kpszsu,False,labeled,nn,593,0,0.0,31566,593,0,64.0,0.000253196607165464,0.10270287378149133,0.0,True\n1746320324,u_now_ua,False,labeled,nn,31032,8656,0.2789378705852023,31071,31032,8656,317.0,1.0,0.0,0.27893787058520236,True\n1305722586,TCH_channel,False,labeled,nn,253,17,0.067193675889328,30098,253,17,291.0,0.006711409395973154,0.24051431988836466,0.06719367588932806,True\n1355310059,espresotb,False,labeled,nn,403,17,0.0421836228287841,30095,403,17,416.0,0.03422381711855396,0.09200558213716109,0.04218362282878412,True\n1105313000,uniannet,False,labeled,nn,107,46,0.4299065420560747,24636,107,46,334.0,0.016397434856725384,0.19384690315772385,0.42990654205607476,True\n1187502427,kherson_news_info,True,labeled,n,648,604,0.9320987654320988,22920,648,604,477.0,0.3269927141049692,0.00017451245582653462,0.9320987654320988,True\n1233777422,UaOnlii,False,labeled,nn,256,88,0.34375,22369,256,88,218.0,0.0024586499776486368,0.1912382655341976,0.34375,True\n1331641097,donbassr,True,labeled,n,138,136,0.9855072463768116,22291,138,136,277.0,0.012561123323314343,8.97223094522453e-05,0.9855072463768116,True\n1122745359,varta1,False,insufficient_data,nn,26,2,0.0769230769230769,21736,26,2,195.0,0.020241052534731806,0.008694452111509798,0.07692307692307693,True\n1199360700,truexanewsua,False,labeled,nn,270,77,0.2851851851851852,19049,270,77,249.0,0.004567168880256181,0.19203107774686334,0.2851851851851852,True\n1352726486,insiderUKR,False,labeled,nn,219,128,0.5844748858447488,18590,219,128,234.0,0.004302925989672977,0.13435886402753872,0.5844748858447488,True\n1058013363,censor_net,False,labeled,nn,84,20,0.238095238095238,18516,84,20,582.0,0.1213545042125729,0.06880535752862389,0.23809523809523808,True\n1470200990,suspilnenews,False,labeled,nn,1011,112,0.1107814045499505,17898,1011,112,258.0,0.1535367080120684,0.24365850933065147,0.11078140454995054,True\n1697250541,truexakyiv,False,labeled,nn,128,20,0.15625,17838,128,20,125.0,0.005491118955566762,0.11873143945761193,0.15625,True\n1059792804,hromadske_ua,False,labeled,nn,55,11,0.2,17664,55,11,370.0,0.0009624094202898551,0.11599864130434782,0.2,True\n1232032465,ukrpravda_news,False,labeled,nn,105,16,0.1523809523809524,17013,105,16,349.0,0.06435876337134125,0.04014341130833431,0.1523809523809524,True\n1200595041,OpenUkraine,True,labeled,n,252,218,0.8650793650793651,15032,252,218,328.0,0.2839391321682504,0.018007841052561632,0.8650793650793651,True\n1281112385,babel,False,insufficient_data,nn,18,2,0.1111111111111111,14704,18,2,543.0,0.0,0.13390914036996734,0.1111111111111111,True\n1210465638,odessa_typical,True,labeled,n,233,168,0.721030042918455,14657,233,168,155.0,0.005457025920873124,0.058867667121418825,0.721030042918455,True\n1280273449,UkraineNow,False,labeled,nn,458,85,0.185589519650655,13899,458,85,463.0,0.039858982660623064,0.14116123462119576,0.185589519650655,True\n1260622817,readovkanews,True,labeled,n,257,215,0.8365758754863813,12848,257,215,762.0,0.020702000155654137,0.03183127091602459,0.8365758754863813,True\n1343028414,ASupersharij,True,labeled,n,74,57,0.7702702702702703,12596,74,57,172.0,0.00015796540557617883,0.02511649948661243,0.7702702702702703,True\n1088461266,ukrinform_news,False,insufficient_data,nn,11,3,0.2727272727272727,12511,11,3,384.0,0.010470785708576453,0.06682119734633522,0.2727272727272727,True\n1578682544,truexakharkiv,False,labeled,nn,100,11,0.11,11949,100,11,132.0,0.009387721192988287,0.19265597740300738,0.11,True\n1495040943,lvivtp,False,insufficient_data,nn,20,0,0.0,11588,20,0,175.0,0.0007766655160510873,0.22833966171901968,0.0,True\n1739446698,otryadkovpaka,True,labeled,n,572,564,0.986013986013986,10760,572,564,731.0,0.24532147489345932,0.02288308319436724,0.986013986013986,True\n1697554072,MonitoringUA,True,labeled,n,524,449,0.8568702290076335,10559,524,449,622.0,0.9883544783185003,0.007100927854572997,0.8568702290076335,True\n1189723223,tokmak_segodna,True,labeled,n,244,221,0.9057377049180328,9584,244,221,221.0,0.9918749357194282,0.0010284891494394734,0.9057377049180327,True\n1456389846,ukr_leaks,True,labeled,n,110,108,0.9818181818181818,9477,110,108,238.0,0.05550279624353698,0.008969083043157118,0.9818181818181818,True\n1496123872,brd_zavtra,True,labeled,n,95,86,0.9052631578947368,9109,95,86,704.0,0.12556250685983975,0.0007683020524640545,0.9052631578947369,True\n1735357542,HersonVestnik,True,labeled,n,198,188,0.9494949494949496,8923,198,188,338.0,0.2468070804391665,0.00011203226529240422,0.9494949494949495,True\n1386691094,torontotv,False,insufficient_data,nn,34,8,0.2352941176470588,8917,34,8,313.0,0.0007850173825277559,0.18145116070427272,0.23529411764705882,True\n1156935944,svoboda_radio,False,labeled,nn,52,11,0.2115384615384615,8868,52,11,535.0,0.0002255299954894001,0.11434370771312584,0.21153846153846154,True\n1601040920,hersonruss,True,labeled,n,668,570,0.8532934131736527,8411,668,570,515.0,0.0023778385447628106,0.007371299488764713,0.8532934131736527,True\n1798592301,melitopol_ru,True,labeled,n,588,435,0.7397959183673469,8326,588,435,508.0,0.0027624309392265192,0.006725918808551525,0.7397959183673469,True\n2009511452,Ukrainerada,False,labeled,nn,55,1,0.0181818181818181,8144,55,1,36.0,0.004343232773799742,0.18159408381265407,0.01818181818181818,True\n1539262259,NowMelitopol,True,labeled,n,573,529,0.9232111692844676,7842,573,529,648.0,0.15517021547877088,0.0010200178503123804,0.9232111692844677,True\n1631471668,herson_today,True,labeled,n,512,500,0.9765625,7396,512,500,867.0,0.12462827791294945,0.0,0.9765625,True\n1052247604,liganet,False,insufficient_data,nn,16,1,0.0625,7193,16,1,455.0,0.00013902405116085083,0.06548032809676074,0.0625,True\n1314050553,suspilnelviv,False,labeled,nn,829,103,0.1242460796139927,6639,829,103,252.0,0.1287844554902847,0.16448260280162674,0.12424607961399277,True\n1312303318,tvdno,False,insufficient_data,nn,3,0,0.0,6612,3,0,359.0,0.021019204597005896,0.04657492817178285,0.0,True\n1754295523,newkahovka_ru,True,labeled,n,420,330,0.7857142857142857,6379,420,330,525.0,0.007838219156607618,0.001254115065057219,0.7857142857142857,True\n1604309069,skadovsk_rus,True,labeled,n,413,335,0.8111380145278451,6168,413,335,535.0,0.008106355382619975,0.0011348897535667965,0.8111380145278451,True\n1764402468,Genichesk_ru,True,labeled,n,414,335,0.8091787439613527,6111,414,335,534.0,0.008345606283750613,0.0013091147111765667,0.8091787439613527,True\n1353955612,novsevlisrub,False,labeled,nn,113,38,0.336283185840708,6103,113,38,592.0,0.024286878565607174,0.06617766911165444,0.336283185840708,True\n1664292906,energodar_ru,True,labeled,n,403,287,0.7121588089330024,5879,403,287,530.0,0.00884504167375404,0.000680387821058003,0.7121588089330024,True\n1217156536,berdyansk_ru,True,labeled,n,399,294,0.7368421052631579,5874,399,294,534.0,0.008512087163772558,0.0006809669731018045,0.7368421052631579,True\n1479946125,ZeRada1,True,labeled,n,146,131,0.8972602739726028,5663,146,131,519.0,0.013413342746205436,0.00017649135192375574,0.8972602739726028,True\n1720948932,polohy_ru,True,labeled,n,390,252,0.6461538461538462,5617,390,252,535.0,0.008901548869503293,0.0005340929321701976,0.6461538461538462,True\n1692721881,zpvestnik,True,labeled,n,65,63,0.9692307692307692,5551,65,63,574.0,0.25562961628535397,0.00468384074941452,0.9692307692307692,True\n1570010567,MediaKiller2021,True,labeled,n,132,132,1.0,4678,132,132,884.0,0.007268063274903805,0.0,1.0,True\n1213194656,netlenkanet,False,insufficient_data,nn,15,7,0.4666666666666667,4665,15,7,666.0,0.46356622374624945,0.1020145735105015,0.4666666666666667,True\n1184797808,Novorossia_Region,True,labeled,n,230,222,0.9652173913043478,4529,230,222,334.0,0.13503971756398941,0.04677846425419241,0.9652173913043478,True\n1287596810,spletnicca,True,labeled,n,252,231,0.9166666666666666,4082,252,231,673.5,0.11513963743263106,0.0,0.9166666666666666,True\n1366496032,KrokooDil,False,insufficient_data,nn,15,8,0.5333333333333333,4008,15,8,113.0,0.004346776141028737,0.0644771794252596,0.5333333333333333,True\n1789573966,zaporojie24,True,labeled,n,437,382,0.8741418764302059,3545,437,382,530.0,0.3749295774647887,0.0016901408450704226,0.8741418764302059,True\n1271765197,rezident_ua,True,labeled,n,111,110,0.990990990990991,3439,111,110,582.0,0.050305321314335565,0.0,0.990990990990991,True\n1664858107,montyan2,False,insufficient_data,nn,6,5,0.8333333333333334,3230,6,5,1238.5,0.10743034055727554,0.0009287925696594427,0.8333333333333334,True\n1441217558,godeye_ru,False,insufficient_data,nn,3,3,1.0,3166,3,3,274.0,0.12590494176896444,0.0003147623544224111,1.0,True\n1368563607,kozytskyy_maksym_official,False,insufficient_data,nn,7,0,0.0,2986,7,0,500.0,0.15030060120240482,0.07748830995323981,0.0,True\n1263569229,vysokygovorit,True,labeled,n,71,60,0.8450704225352113,2922,71,60,635.0,0.4805061559507524,0.016073871409028728,0.8450704225352113,True\n1278514229,htosho,False,insufficient_data,nn,2,2,1.0,2917,2,2,134.0,0.038991596638655465,0.02857142857142857,1.0,True\n1202009667,nach_shtabu,False,labeled,nn,64,33,0.515625,2516,64,33,165.0,0.004352987732489117,0.05698456667985754,0.515625,True\n1242259852,legitimniy,True,labeled,n,70,70,1.0,2270,70,70,690.0,0.02158590308370044,0.06211453744493392,1.0,True\n1447182889,ZE_kartel,False,insufficient_data,nn,42,42,1.0,2085,42,42,1293.0,0.05081495685522531,0.0052732502396931925,1.0,True\n1176247089,harry_homolsky,False,insufficient_data,nn,5,5,1.0,1906,5,5,367.0,0.1195822454308094,0.0005221932114882506,1.0,True\n1207476194,skosoi,False,insufficient_data,nn,40,40,1.0,1630,40,40,1451.0,0.030061349693251534,0.0049079754601227,1.0,True\n1663948542,vasulivka,False,insufficient_data,nn,39,35,0.8974358974358975,1537,39,35,494.0,0.0987012987012987,0.012337662337662338,0.8974358974358975,True\n1514326454,Sharij_News_l,False,insufficient_data,nn,6,6,1.0,1424,6,6,239.0,0.0,0.0,1.0,True\n1209080275,cherniy_kvartal,True,labeled,n,120,109,0.9083333333333332,1301,120,109,323.0,0.0007686395080707148,0.023059185242121444,0.9083333333333333,True\n1678485342,depzdravzo,True,labeled,n,348,316,0.9080459770114944,1249,348,316,943.0,0.2547923322683706,0.003194888178913738,0.9080459770114943,True\n1494993138,tsibulya_ua,False,insufficient_data,nn,12,10,0.8333333333333334,783,12,10,566.0,0.005108556832694764,0.001277139208173691,0.8333333333333334,True\n2095513566,melitopoladmln,True,labeled,n,92,78,0.8478260869565217,775,92,78,341.0,0.0,0.0,0.8478260869565217,True\n1240977961,OlgaSharij,False,insufficient_data,nn,16,13,0.8125,619,16,13,136.0,0.03503184713375796,0.0,0.8125,True\n1788865698,Pukhov_M,True,labeled,n,290,242,0.8344827586206897,562,290,242,344.0,0.1566265060240964,0.0378657487091222,0.8344827586206897,True\n1605512487,ombr_63,False,no_labeled_messages,nn,0,0,,476,0,0,349.5,0.037815126050420166,0.0,,True\n2004911807,tokmaksegodnya,False,insufficient_data,nn,17,14,0.8235294117647058,417,17,14,872.0,0.49640287769784175,0.0,0.8235294117647058,True\n1136914098,buntariy,True,labeled,n,75,63,0.84,353,75,63,199.0,0.025495750708215296,0.0,0.84,True\n1585880158,er_energodar,True,labeled,n,105,97,0.923809523809524,322,105,97,755.0,0.7608695652173914,0.0,0.9238095238095239,True\n2421064536,gorsovet_energodar,True,labeled,n,116,111,0.956896551724138,181,116,111,825.0,0.20994475138121546,0.0055248618784530384,0.9568965517241379,True\n1236564472,Ukrainian_format,False,insufficient_data,nn,1,1,1.0,155,1,1,153.0,0.07692307692307693,0.08974358974358974,1.0,True\n1285869345,life_odessa,False,insufficient_data,nn,3,1,0.3333333333333333,141,3,1,129.0,0.0,0.02127659574468085,0.3333333333333333,True\n1173366996,photographermilitary,True,labeled,n,114,112,0.9824561403508772,121,114,112,278.0,0.9672131147540983,0.0,0.9824561403508771,True\n1317786502,maksnazar,False,insufficient_data,nn,1,1,1.0,112,1,1,130.0,0.02564102564102564,0.0,1.0,True\n1195323522,Ten_NaPleten,False,insufficient_data,nn,19,13,0.6842105263157895,112,19,13,653.0,0.0,0.0,0.6842105263157895,True\n1727557197,smertvorogy,False,no_labeled_messages,nn,0,0,,104,0,0,533.0,0.11538461538461539,0.0,,True\n2438139679,polititionrasklad,False,insufficient_data,nn,1,0,0.0,100,1,0,396.5,0.7128712871287128,0.12871287128712872,0.0,True\n1689116658,energadartoday,False,insufficient_data,nn,30,28,0.9333333333333332,41,30,28,1096.0,0.1951219512195122,0.0,0.9333333333333333,True\n1762050427,atomgradenergodar,False,insufficient_data,nn,30,28,0.9333333333333332,37,30,28,1049.0,0.8378378378378378,0.0,0.9333333333333333,True\n1200907961,herson_live,False,no_labeled_messages,nn,0,0,,26,0,0,127.5,0.0,0.0,,True\n1522487464,Kharkiv_antiFA,False,insufficient_data,nn,1,1,1.0,5,1,1,193.0,0.4,0.0,1.0,True\n1150201825,kharkov_trempel,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1287246088,zp_live,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1240111734,hudoimir,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1237733774,SBUmole,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1333997861,sharij_official,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1471027234,niklive,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1405303803,ukraina24tv,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1667648011,HolodniyYar,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1709600040,novgorodseversky_ru,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1722183049,news24tvua,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1745404668,truexadnipro,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n1778913067,yug_plazdarm,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n2054787161,realnaukraine,False,no_labeled_messages,nn,0,0,,0,0,0,,,,,True\n', 'data_small/channel_flow_edges.csv': 'src_channel_id,src_channel,dst_channel_id,dst_channel,weight\n1197363285,u_now,1746320324,u_now_ua,30810\n1470200990,suspilnenews,1314050553,suspilnelviv,712\n1101170442,rian_ru,1697554072,MonitoringUA,380\n1678485342,depzdravzo,1539262259,NowMelitopol,330\n1788865698,Pukhov_M,1789573966,zaporojie24,266\n1260622817,readovkanews,1189723223,tokmak_segodna,146\n1187502427,kherson_news_info,1631471668,herson_today,117\n1101170442,rian_ru,1173366996,photographermilitary,114\n1223955273,kpszsu,1280273449,UkraineNow,113\n1331641097,donbassr,1175084215,ukr_2025_ru,106\n1314050553,suspilnelviv,1470200990,suspilnenews,103\n2421064536,gorsovet_energodar,1789573966,zaporojie24,82\n1735357542,HersonVestnik,1631471668,herson_today,74\n1101170442,rian_ru,1187502427,kherson_news_info,65\n1601040920,hersonruss,1631471668,herson_today,59\n1570010567,MediaKiller2021,1287596810,spletnicca,55\n1735357542,HersonVestnik,1187502427,kherson_news_info,50\n1199360700,truexanewsua,1697250541,truexakyiv,48\n1601040920,hersonruss,1754295523,newkahovka_ru,45\n1199360700,truexanewsua,1578682544,truexakharkiv,45\n1601040920,hersonruss,1764402468,Genichesk_ru,44\n1601040920,hersonruss,1604309069,skadovsk_rus,44\n1479946125,ZeRada1,1287596810,spletnicca,43\n2421064536,gorsovet_energodar,1585880158,er_energodar,42\n1570010567,MediaKiller2021,1271765197,rezident_ua,41\n1798592301,melitopol_ru,1664292906,energodar_ru,41\n1798592301,melitopol_ru,1217156536,berdyansk_ru,41\n1798592301,melitopol_ru,1720948932,polohy_ru,40\n1187502427,kherson_news_info,1184797808,Novorossia_Region,39\n1287596810,spletnicca,1479946125,ZeRada1,36\n1263569229,vysokygovorit,1697554072,MonitoringUA,34\n1788865698,Pukhov_M,1585880158,er_energodar,31\n1175084215,ukr_2025_ru,1187502427,kherson_news_info,26\n1689116658,energadartoday,1762050427,atomgradenergodar,25\n1585880158,er_energodar,1789573966,zaporojie24,21\n1631471668,herson_today,1187502427,kherson_news_info,19\n1271765197,rezident_ua,1479946125,ZeRada1,19\n1739446698,otryadkovpaka,1456389846,ukr_leaks,19\n1601040920,hersonruss,1187502427,kherson_news_info,18\n1692721881,zpvestnik,1789573966,zaporojie24,17\n1195323522,Ten_NaPleten,1287596810,spletnicca,17\n1447182889,ZE_kartel,1287596810,spletnicca,15\n1260622817,readovkanews,1798592301,melitopol_ru,14\n1631471668,herson_today,1735357542,HersonVestnik,14\n1585880158,er_energodar,2421064536,gorsovet_energodar,14\n1263569229,vysokygovorit,1189723223,tokmak_segodna,14\n1343028414,ASupersharij,1240977961,OlgaSharij,14\n1184797808,Novorossia_Region,1187502427,kherson_news_info,12\n1260622817,readovkanews,1601040920,hersonruss,12\n1175084215,ukr_2025_ru,1331641097,donbassr,12\n1735357542,HersonVestnik,1184797808,Novorossia_Region,12\n1101170442,rian_ru,1735357542,HersonVestnik,10\n1175084215,ukr_2025_ru,1456389846,ukr_leaks,10\n1479946125,ZeRada1,1697554072,MonitoringUA,10\n1242259852,legitimniy,1697554072,MonitoringUA,9\n1207476194,skosoi,1287596810,spletnicca,9\n1223955273,kpszsu,2009511452,Ukrainerada,9\n1242259852,legitimniy,1447182889,ZE_kartel,8\n1242259852,legitimniy,1189723223,tokmak_segodna,8\n1287596810,spletnicca,1271765197,rezident_ua,8\n1101170442,rian_ru,1184797808,Novorossia_Region,7\n1101170442,rian_ru,1189723223,tokmak_segodna,7\n1260622817,readovkanews,1692721881,zpvestnik,7\n1175084215,ukr_2025_ru,1200595041,OpenUkraine,7\n1242259852,legitimniy,1287596810,spletnicca,7\n1101170442,rian_ru,1631471668,herson_today,6\n1175084215,ukr_2025_ru,1739446698,otryadkovpaka,6\n1539262259,NowMelitopol,1496123872,brd_zavtra,6\n1789573966,zaporojie24,1788865698,Pukhov_M,6\n1242259852,legitimniy,1207476194,skosoi,6\n1343028414,ASupersharij,1697554072,MonitoringUA,6\n1271765197,rezident_ua,1287596810,spletnicca,6\n1287596810,spletnicca,1189723223,tokmak_segodna,6\n1101170442,rian_ru,1200595041,OpenUkraine,5\n1175084215,ukr_2025_ru,1697554072,MonitoringUA,5\n1175084215,ukr_2025_ru,1735357542,HersonVestnik,5\n1762050427,atomgradenergodar,1689116658,energadartoday,5\n1242259852,legitimniy,1187502427,kherson_news_info,5\n1271765197,rezident_ua,1447182889,ZE_kartel,5\n1479946125,ZeRada1,1271765197,rezident_ua,5\n1287596810,spletnicca,1697554072,MonitoringUA,5\n1447182889,ZE_kartel,1271765197,rezident_ua,5\n1601040920,hersonruss,1798592301,melitopol_ru,5\n1101170442,rian_ru,1175084215,ukr_2025_ru,4\n1101170442,rian_ru,1263569229,vysokygovorit,4\n1260622817,readovkanews,1217156536,berdyansk_ru,4\n1260622817,readovkanews,1187502427,kherson_news_info,4\n1260622817,readovkanews,1764402468,Genichesk_ru,4\n1260622817,readovkanews,1664292906,energodar_ru,4\n1175084215,ukr_2025_ru,1788865698,Pukhov_M,4\n1754295523,newkahovka_ru,1631471668,herson_today,4\n1678485342,depzdravzo,1789573966,zaporojie24,4\n1692721881,zpvestnik,2004911807,tokmaksegodnya,4\n1788865698,Pukhov_M,2421064536,gorsovet_energodar,4\n1343028414,ASupersharij,1189723223,tokmak_segodna,4\n1271765197,rezident_ua,1189723223,tokmak_segodna,4\n1271765197,rezident_ua,1207476194,skosoi,4\n1200595041,OpenUkraine,1739446698,otryadkovpaka,4\n1456389846,ukr_leaks,1200595041,OpenUkraine,4\n1570010567,MediaKiller2021,1447182889,ZE_kartel,4\n1764402468,Genichesk_ru,1187502427,kherson_news_info,4\n1101170442,rian_ru,1496123872,brd_zavtra,3\n1260622817,readovkanews,1604309069,skadovsk_rus,3\n1260622817,readovkanews,1754295523,newkahovka_ru,3\n1260622817,readovkanews,1720948932,polohy_ru,3\n1678485342,depzdravzo,1692721881,zpvestnik,3\n1692721881,zpvestnik,1678485342,depzdravzo,3\n1692721881,zpvestnik,1539262259,NowMelitopol,3\n1789573966,zaporojie24,1585880158,er_energodar,3\n1263569229,vysokygovorit,1663948542,vasulivka,3\n1200595041,OpenUkraine,1697554072,MonitoringUA,3\n1601040920,hersonruss,1217156536,berdyansk_ru,3\n1601040920,hersonruss,1720948932,polohy_ru,3\n1601040920,hersonruss,1664292906,energodar_ru,3\n1739446698,otryadkovpaka,1189723223,tokmak_segodna,3\n1184797808,Novorossia_Region,1735357542,HersonVestnik,2\n1101170442,rian_ru,1678485342,depzdravzo,2\n1260622817,readovkanews,1697554072,MonitoringUA,2\n1539262259,NowMelitopol,1735357542,HersonVestnik,2\n1539262259,NowMelitopol,1631471668,herson_today,2\n1735357542,HersonVestnik,1496123872,brd_zavtra,2\n1789573966,zaporojie24,2421064536,gorsovet_energodar,2\n1788865698,Pukhov_M,1692721881,zpvestnik,2\n1217156536,berdyansk_ru,1720948932,polohy_ru,2\n1217156536,berdyansk_ru,1664292906,energodar_ru,2\n1242259852,legitimniy,1136914098,buntariy,2\n1242259852,legitimniy,1570010567,MediaKiller2021,2\n1664858107,montyan2,1200595041,OpenUkraine,2\n1479946125,ZeRada1,1184797808,Novorossia_Region,2\n1479946125,ZeRada1,1136914098,buntariy,2\n1287596810,spletnicca,1184797808,Novorossia_Region,2\n1240977961,OlgaSharij,1343028414,ASupersharij,2\n1570010567,MediaKiller2021,1479946125,ZeRada1,2\n1570010567,MediaKiller2021,1184797808,Novorossia_Region,2\n1207476194,skosoi,1271765197,rezident_ua,2\n1210465638,odessa_typical,1200595041,OpenUkraine,2\n1601040920,hersonruss,1260622817,readovkanews,2\n1176247089,harry_homolsky,1735357542,HersonVestnik,2\n1764402468,Genichesk_ru,1601040920,hersonruss,2\n1764402468,Genichesk_ru,1739446698,otryadkovpaka,2\n1798592301,melitopol_ru,1601040920,hersonruss,2\n1798592301,melitopol_ru,1260622817,readovkanews,2\n1195323522,Ten_NaPleten,1479946125,ZeRada1,2\n1232032465,ukrpravda_news,1058013363,censor_net,2\n1199360700,truexanewsua,1189723223,tokmak_segodna,2\n1101170442,rian_ru,1739446698,otryadkovpaka,1\n1101170442,rian_ru,1456389846,ukr_leaks,1\n1101170442,rian_ru,1331641097,donbassr,1\n1101170442,rian_ru,1663948542,vasulivka,1\n1101170442,rian_ru,1789573966,zaporojie24,1\n1260622817,readovkanews,1788865698,Pukhov_M,1\n1260622817,readovkanews,1789573966,zaporojie24,1\n1260622817,readovkanews,1200595041,OpenUkraine,1\n1175084215,ukr_2025_ru,1176247089,harry_homolsky,1\n1175084215,ukr_2025_ru,1663948542,vasulivka,1\n1175084215,ukr_2025_ru,1789573966,zaporojie24,1\n1539262259,NowMelitopol,1789573966,zaporojie24,1\n1735357542,HersonVestnik,1456389846,ukr_leaks,1\n1754295523,newkahovka_ru,1604309069,skadovsk_rus,1\n1754295523,newkahovka_ru,1764402468,Genichesk_ru,1\n1754295523,newkahovka_ru,1601040920,hersonruss,1\n1604309069,skadovsk_rus,1631471668,herson_today,1\n1496123872,brd_zavtra,1735357542,HersonVestnik,1\n1692721881,zpvestnik,1585880158,er_energodar,1\n1585880158,er_energodar,1788865698,Pukhov_M,1\n1217156536,berdyansk_ru,1798592301,melitopol_ru,1\n1242259852,legitimniy,1271765197,rezident_ua,1\n1242259852,legitimniy,1522487464,Kharkiv_antiFA,1\n1343028414,ASupersharij,1213194656,netlenkanet,1\n1343028414,ASupersharij,2438139679,polititionrasklad,1\n1271765197,rezident_ua,1697554072,MonitoringUA,1\n1271765197,rezident_ua,1242259852,legitimniy,1\n1479946125,ZeRada1,1189723223,tokmak_segodna,1\n1479946125,ZeRada1,1187502427,kherson_news_info,1\n1200595041,OpenUkraine,1189723223,tokmak_segodna,1\n1456389846,ukr_leaks,1697554072,MonitoringUA,1\n1287596810,spletnicca,1207476194,skosoi,1\n1287596810,spletnicca,1447182889,ZE_kartel,1\n1447182889,ZE_kartel,1207476194,skosoi,1\n1447182889,ZE_kartel,1189723223,tokmak_segodna,1\n1570010567,MediaKiller2021,1189723223,tokmak_segodna,1\n1570010567,MediaKiller2021,1207476194,skosoi,1\n1312303318,tvdno,1697554072,MonitoringUA,1\n1207476194,skosoi,1447182889,ZE_kartel,1\n1210465638,odessa_typical,1187502427,kherson_news_info,1\n1210465638,odessa_typical,1697554072,MonitoringUA,1\n1210465638,odessa_typical,1184797808,Novorossia_Region,1\n1210465638,odessa_typical,1175084215,ukr_2025_ru,1\n1601040920,hersonruss,1184797808,Novorossia_Region,1\n1601040920,hersonruss,1735357542,HersonVestnik,1\n1739446698,otryadkovpaka,1496123872,brd_zavtra,1\n1739446698,otryadkovpaka,1735357542,HersonVestnik,1\n1176247089,harry_homolsky,1697554072,MonitoringUA,1\n1176247089,harry_homolsky,1456389846,ukr_leaks,1\n1764402468,Genichesk_ru,1754295523,newkahovka_ru,1\n1764402468,Genichesk_ru,1631471668,herson_today,1\n1764402468,Genichesk_ru,1604309069,skadovsk_rus,1\n1236564472,Ukrainian_format,1456389846,ukr_leaks,1\n1798592301,melitopol_ru,1189723223,tokmak_segodna,1\n1798592301,melitopol_ru,1539262259,NowMelitopol,1\n1223955273,kpszsu,1305722586,TCH_channel,1\n1232032465,ukrpravda_news,1697250541,truexakyiv,1\n1232032465,ukrpravda_news,1578682544,truexakharkiv,1\n1232032465,ukrpravda_news,1199360700,truexanewsua,1\n1232032465,ukrpravda_news,1317786502,maksnazar,1\n1470200990,suspilnenews,1386691094,torontotv,1\n1386691094,torontotv,1059792804,hromadske_ua,1\n1352726486,insiderUKR,1189723223,tokmak_segodna,1\n1105313000,uniannet,1199360700,truexanewsua,1\n1105313000,uniannet,1058013363,censor_net,1\n1059792804,hromadske_ua,1386691094,torontotv,1\n1059792804,hromadske_ua,1746320324,u_now_ua,1\n1059792804,hromadske_ua,1355310059,espresotb,1\n1355310059,espresotb,1059792804,hromadske_ua,1\n'}

for rel, text in FILES.items():
    p = REPO / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
    print(f"{hashlib.sha256(text.encode()).hexdigest()[:12]}  {rel}")


In [ ]:
# 3. Золотий набір: клон TeleNarratives (потрібен лише golden set CSV)
EXT = REPO / "external" / "TeleNarratives"
if not EXT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yuliavistak/TeleNarratives", str(EXT)], check=True)
head = subprocess.run(["git", "-C", str(EXT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("TeleNarratives commit:", head)
gold_csv = next(EXT.glob("labeling results/strong labeling/golden set (labeled).csv"))
print("Золотий набір:", gold_csv)


In [ ]:
# 4. Перевірка, що потрібні файли на місці
for n in ["telegram_messages.parquet", "narratives_hierarchy.csv"]:
    assert (RAW / n).exists(), f"немає {n}"
msgs_p = sorted(RAW.rglob("messages.parquet"))
assert msgs_p, "немає messages.parquet у розпакованому parquet-наборі"
gd = msgs_p[0].parent
for n in ["forward_edges.parquet", "semantic_edges.parquet", "narrative_subnarrative_coding_en.csv"]:
    assert (gd / n).exists(), f"немає {n} поруч із messages.parquet"
print("Набір:", gd)
try:
    import psutil
    print(f"Пам'ять: {psutil.virtual_memory().total / 1e9:.1f} ГБ усього, {psutil.virtual_memory().available / 1e9:.1f} ГБ вільно")
except ImportError:
    pass


In [ ]:
# 5. Запуск S4 (скрипт scripts/s04_semantic.py з репозиторію)
cmd = [sys.executable, "-I", str(REPO / "scripts" / "s04_semantic.py"), "--raw-dir", str(RAW), "--out", str(OUT),
       "--interim", str(INTERIM), "--golden", str(gold_csv), "--batch-size", "64", "--max-len", "512", "--k", "80",
       "--umap-n", "20000"]
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(REPO))
for line in proc.stdout:
    print(line, end="")
code = proc.wait()
print(f"\nКод завершення {code}, {time.time() - t0:.0f} с")


In [ ]:
# 6. Підсумок і архів
import shutil

checks = json.loads((OUT / "checks.json").read_text(encoding="utf-8"))
shutil.make_archive(str(OUT.parent / "04_semantic"), "zip", OUT)
print("Архів:", OUT.parent / "04_semantic.zip")

print("\nПеревірки:")
for c in checks:
    mark = "ok" if c["passed"] else ("ПОМИЛКА" if c["hard"] else "розбіжність")
    print(f"  [{mark}] {c['check']}: {c['detail']}")
print("\nФайли в", OUT)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  {p.stat().st_size / 1e3:.1f} КБ")
assert code == 0, "S4: жорсткі перевірки не пройдено, див. вивід вище"
